# EUR Inflation Pricer — learning notebook

Learning path through the EUR inflation market, using live Bloomberg data:

1. Bloomberg API connectivity
2. Market data: CPI fixings, ZC inflation swaps, €STR OIS curve
3. Inflation curve building (pillars, lag, seasonality)
4. Inflation swap pricer (ZC / YoY)
5. Linker pricer (index ratio, real yield, breakeven)
6. Asset swap (ASW) pricer
7. Inflation options pricer (ZC / YoY caps & floors)


## 0. Setup

In [ ]:
import socket

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

def add_date_vline(fig, x, text):
    """Vertical dotted line + label at a date (plotly's add_vline annotation fails on Timestamps)."""
    fig.add_vline(x=x, line_dash="dot", line_color="gray")
    fig.add_annotation(x=x, y=1, yref="paper", text=text, showarrow=False, xanchor="left", yanchor="bottom")

## 1. Bloomberg API connectivity

Self-contained (no project imports). Requires a logged-in Bloomberg terminal, `xbbg` and `pyarrow >= 22`.

Two checks before any request, otherwise calls can block forever:
1. **bbcomm** listening on `127.0.0.1:8194` (fast socket probe)
2. **`//blp/refdata`** service actually opens (fails if the terminal is not logged in)

`bdp` / `bdh` wrappers below pivot xbbg's long output (`ticker | field | value`) into wide DataFrames.

In [ ]:
import blpapi
from xbbg import blp

blp.set_backend("pandas")  # return pandas DataFrames (still needs pyarrow >= 22 internally)

BBG_HOST, BBG_PORT = "127.0.0.1", 8194


def bbcomm_up(timeout_s: float = 1.0) -> bool:
    """Fast check that bbcomm is listening."""
    try:
        with socket.create_connection((BBG_HOST, BBG_PORT), timeout=timeout_s):
            return True
    except OSError:
        return False


def refdata_ready(timeout_s: float = 10.0) -> bool:
    """True if the //blp/refdata service opens within `timeout_s`."""
    opts = blpapi.SessionOptions()
    opts.setServerHost(BBG_HOST)
    opts.setServerPort(BBG_PORT)
    session = blpapi.Session(opts)
    try:
        if not session.start():
            return False
        session.openServiceAsync("//blp/refdata", blpapi.CorrelationId(1))
        waited_ms = 0
        while waited_ms < timeout_s * 1000:
            ev = session.nextEvent(500)
            waited_ms += 500
            for msg in ev:
                if msg.messageType() == blpapi.Name("ServiceOpened"):
                    return True
                if msg.messageType() == blpapi.Name("ServiceOpenFailure"):
                    return False
        return False
    finally:
        session.stop()


def check_bloomberg() -> bool:
    if not bbcomm_up():
        print(f"bbcomm not reachable on {BBG_HOST}:{BBG_PORT} -> start the Bloomberg terminal.")
        return False
    if not refdata_ready():
        print("bbcomm is up but //blp/refdata did not open -> log into the Bloomberg terminal.")
        return False
    print("Bloomberg ready.")
    return True


BBG_READY = check_bloomberg()

In [ ]:
def bdp(tickers, fields, numeric=()) -> pd.DataFrame:
    """Reference data -> one row per ticker, one column per field."""
    if not BBG_READY:
        raise RuntimeError("Bloomberg not ready: rerun check_bloomberg().")
    raw = blp.bdp(tickers=list(tickers), flds=list(fields))
    df = raw.pivot(index="ticker", columns="field", values="value")
    df = df.reindex(index=list(tickers), columns=[f for f in fields if f in df.columns])
    for f in numeric:
        if f in df:
            df[f] = pd.to_numeric(df[f], errors="coerce")
    return df


def bdh(tickers, field, start, end) -> pd.DataFrame:
    """Historical data for one field -> date x ticker."""
    if not BBG_READY:
        raise RuntimeError("Bloomberg not ready: rerun check_bloomberg().")
    raw = blp.bdh(tickers=list(tickers), flds=field, start_date=start, end_date=end)
    raw["value"] = pd.to_numeric(raw["value"], errors="coerce")
    raw["date"] = pd.to_datetime(raw["date"])
    hist = raw.pivot(index="date", columns="ticker", values="value")
    return hist[[t for t in tickers if t in hist.columns]]

## 2. Market data

### 2.1 EUR zero-coupon inflation swaps (HICPxT)

Par rates of the EUR zero-coupon inflation swap on Eurozone HICP ex-tobacco (`CPTFEMU Index`), maturity *n* years.

| Source | Ticker | Notes |
|---|---|---|
| `"ICAP"` (default) | `EUSWI<n> ICPI Curncy` | broker (ICAP) pages, more accurate |
| `"BBG"` | `EUSWI<n> Curncy` | Bloomberg generic composite |

With `FALLBACK_TO_BBG = True`, any tenor missing on ICAP (no `PX_LAST`) is filled from the generic Bloomberg ticker;
the `SOURCE` column shows where each rate comes from.

Contract recap: at maturity $T$ the inflation receiver gets $N\left(\frac{I(T-3m)}{I(0-3m)} - 1\right)$ and pays
$N\left((1+K)^T - 1\right)$. The index uses a 3-month lag, monthly fixings, no interpolation.

In [ ]:
# ---- Source selection ------------------------------------------------------
INF_SWAP_SOURCE = "ICAP"     # "ICAP" (broker) or "BBG" (generic Bloomberg)
FALLBACK_TO_BBG = True       # fill tenors missing on ICAP from the generic BBG ticker
# -----------------------------------------------------------------------------

INF_SWAP_TENORS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 20, 25, 30]
INF_SWAP_SOURCES = {
    "ICAP": "EUSWI{n} ICPI Curncy",
    "BBG": "EUSWI{n} Curncy",
}


def inf_swap_ticker(tenor: int, source: str) -> str:
    return INF_SWAP_SOURCES[source].format(n=tenor)

In [ ]:
SWAP_FIELDS = ["NAME", "PX_BID", "PX_ASK", "PX_LAST", "LAST_UPDATE_DT"]
SWAP_NUMERIC = ["PX_BID", "PX_ASK", "PX_LAST"]


def _swap_snapshot(tenors, source) -> pd.DataFrame:
    tickers = [inf_swap_ticker(t, source) for t in tenors]
    df = bdp(tickers, SWAP_FIELDS, numeric=SWAP_NUMERIC).reset_index()
    df.insert(0, "tenor_y", list(tenors))
    df["SOURCE"] = source
    return df.set_index("tenor_y")


def get_inflation_swaps(source=INF_SWAP_SOURCE, fallback=FALLBACK_TO_BBG) -> pd.DataFrame:
    """Snapshot of the EUR ZC inflation swap curve, one row per tenor (rates in %)."""
    df = _swap_snapshot(INF_SWAP_TENORS, source)
    missing = df.index[df["PX_LAST"].isna()].tolist()
    if fallback and source != "BBG" and missing:
        print(f"{source} missing for tenors {missing} -> using BBG generic tickers.")
        df.loc[missing] = _swap_snapshot(missing, "BBG")
    df["LAST_UPDATE_DT"] = pd.to_datetime(df["LAST_UPDATE_DT"], errors="coerce")
    df["MID"] = df[["PX_BID", "PX_ASK"]].mean(axis=1).fillna(df["PX_LAST"])
    df["BID_ASK_BP"] = (df["PX_ASK"] - df["PX_BID"]) * 100
    return df


inf_swaps = get_inflation_swaps()
inf_swaps

Quick comparison ICAP vs generic Bloomberg (mid, bp difference):

In [ ]:
cmp = pd.DataFrame({
    "ICAP": get_inflation_swaps("ICAP", fallback=False)["MID"],
    "BBG": get_inflation_swaps("BBG")["MID"],
})
cmp["ICAP-BBG (bp)"] = (cmp["ICAP"] - cmp["BBG"]) * 100
cmp.round(4)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=inf_swaps.index, y=inf_swaps["PX_ASK"], mode="lines", line=dict(width=0),
    showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(
    x=inf_swaps.index, y=inf_swaps["PX_BID"], mode="lines", line=dict(width=0),
    fill="tonexty", fillcolor="rgba(99,110,250,0.2)", name="bid/ask"))
fig.add_trace(go.Scatter(
    x=inf_swaps.index, y=inf_swaps["MID"], mode="lines+markers", name="mid",
    customdata=inf_swaps[["ticker", "SOURCE", "BID_ASK_BP"]],
    hovertemplate="%{x}Y: %{y:.4f}%<br>%{customdata[0]} (%{customdata[1]})"
                  "<br>bid/ask %{customdata[2]:.1f} bp<extra></extra>"))
fig.update_layout(title=f"EUR ZC inflation swap curve (HICPxT) - source {INF_SWAP_SOURCE}",
                  xaxis_title="Maturity (years)", yaxis_title="ZC rate (%)",
                  hovermode="x unified", height=450)
fig.show()

### 2.2 Inflation swap history

In [ ]:
HIST_END = pd.Timestamp.today()
HIST_START = HIST_END - pd.DateOffset(years=1)


def get_inflation_swap_history(tenors=(2, 5, 10, 30), start=HIST_START, end=HIST_END,
                               source=INF_SWAP_SOURCE, fallback=FALLBACK_TO_BBG) -> pd.DataFrame:
    """Daily PX_LAST history, one column per tenor (rates in %)."""
    s, e = start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")
    tickers = {f"{t}Y": inf_swap_ticker(t, source) for t in tenors}
    hist = bdh(list(tickers.values()), "PX_LAST", s, e)
    hist = hist.rename(columns={tk: col for col, tk in tickers.items()}).reindex(columns=list(tickers))
    missing = [col for col in hist.columns if hist[col].isna().all()]
    if fallback and source != "BBG" and missing:
        print(f"{source} history missing for {missing} -> using BBG generic tickers.")
        bbg_t = {col: inf_swap_ticker(int(col[:-1]), "BBG") for col in missing}
        alt = bdh(list(bbg_t.values()), "PX_LAST", s, e)
        for col, tk in bbg_t.items():
            if tk in alt:
                hist[col] = alt[tk]
    return hist


inf_swaps_hist = get_inflation_swap_history()
inf_swaps_hist.tail()

In [ ]:
fig = px.line(inf_swaps_hist, labels={"value": "ZC rate (%)", "date": "", "ticker": "tenor"},
              title=f"EUR ZC inflation swaps - 1Y history, source {INF_SWAP_SOURCE}")
fig.update_layout(hovermode="x unified", height=450, legend_title_text="tenor")
fig.show()

### 2.3 HICPxT fixings (published CPI)

`CPTFEMU Index` = Eurostat Eurozone HICP ex-tobacco, non-seasonally adjusted, monthly. This is the index referenced by
EUR inflation swaps and by OAT€i / Bund€i / BTP€i linkers. Bloomberg dates each fixing at month-end; below it is
indexed by **reference month** (`Period('M')`).

In [ ]:
CPI_TICKER = "CPTFEMU Index"
CPI_HIST_START = "2010-01-01"


def get_cpi_fixings(ticker=CPI_TICKER, start=CPI_HIST_START) -> pd.Series:
    """Published monthly index levels, indexed by reference month."""
    hist = bdh([ticker], "PX_LAST", start, pd.Timestamp.today().strftime("%Y-%m-%d"))
    s = hist[ticker].dropna()
    s.index = s.index.to_period("M")
    s.name = ticker
    return s


cpi_fixings = get_cpi_fixings()
cpi_info = bdp([CPI_TICKER], ["NAME", "PX_LAST", "OBSERVATION_PERIOD", "ECO_RELEASE_DT"])
LAST_FIXING_MONTH = cpi_fixings.index[-1]

print(f"Last published fixing : {LAST_FIXING_MONTH} = {cpi_fixings.iloc[-1]:.2f}")
print(f"Next release          : {cpi_info.loc[CPI_TICKER, 'OBSERVATION_PERIOD']} "
      f"on {cpi_info.loc[CPI_TICKER, 'ECO_RELEASE_DT']}")

cpi_table = pd.DataFrame({
    "INDEX": cpi_fixings,
    "MOM_PCT": cpi_fixings.pct_change() * 100,
    "YOY_PCT": cpi_fixings.pct_change(12) * 100,
})
cpi_table.tail(13).round(3)

### 2.4 Short end: HICPxT fixing swaps (BGC broker)

Monthly fixing swaps quoted by BGC on Bloomberg (`INFS` pricing source):

| Ticker | Fixing month | Example |
|---|---|---|
| `EUSWIF<m> INFS Curncy` | next (1st) fixing of calendar month *m* | `EUSWIF10` = Oct-26 |
| `EUSWIH<m> INFS Curncy` | 2nd fixing of calendar month *m* (one year later) | `EUSWIH10` = Oct-27 |

- **Fixing month** = ticker `MATURITY` − 3 months (payment ~3 months after the reference month).
- **Quote** = expected **YoY HICPxT inflation of the fixing month, in bp** (e.g. 396 → 3.96%).
- **Implied index**: $I(m) = I(m-12)\,\left(1 + \frac{q_m}{10\,000}\right)$, with $I(m-12)$ the published fixing if
  available, otherwise the previously implied level (the 2nd-year `H` contracts chain off the `F` ones).

These pin down the monthly CPI path (including seasonality) for roughly the next two years, which is exactly
where the annual ZC swap pillars are too coarse.

In [ ]:
# ---- Source selection ------------------------------------------------------
FIXING_SOURCE = "BGC"
FIXING_SWAP_SOURCES = {
    "BGC": "EUSWI{yr}{m} INFS Curncy",   # yr: F = 1st fixing, H = 2nd fixing
}
# -----------------------------------------------------------------------------

FIXING_FIELDS = ["NAME", "PX_BID", "PX_ASK", "PX_LAST", "MATURITY", "LAST_UPDATE_DT"]


def get_fixing_swaps(source=FIXING_SOURCE, cpi=None) -> pd.DataFrame:
    """Fixing swap quotes + implied monthly HICPxT path, indexed by fixing month."""
    cpi = cpi_fixings if cpi is None else cpi
    tickers = [FIXING_SWAP_SOURCES[source].format(yr=yr, m=m) for yr in "FH" for m in range(1, 13)]
    df = bdp(tickers, FIXING_FIELDS, numeric=["PX_BID", "PX_ASK", "PX_LAST"]).reset_index()
    df["MATURITY"] = pd.to_datetime(df["MATURITY"], errors="coerce")
    df["LAST_UPDATE_DT"] = pd.to_datetime(df["LAST_UPDATE_DT"], errors="coerce")
    df["FIXING_MONTH"] = (df["MATURITY"] - pd.DateOffset(months=3)).dt.to_period("M")
    df["SOURCE"] = source
    df["MID_BP"] = df[["PX_BID", "PX_ASK"]].mean(axis=1).fillna(df["PX_LAST"])
    df["YOY_PCT"] = df["MID_BP"] / 100
    df = df.set_index("FIXING_MONTH").sort_index()

    # Chain implied index levels: published fixing if known, else previously implied level.
    levels = cpi.copy()
    implied = {}
    for month, row in df.iterrows():
        base = levels.get(month - 12)
        if base is None or pd.isna(row["MID_BP"]):
            implied[month] = np.nan
            continue
        implied[month] = base * (1 + row["MID_BP"] / 10_000)
        levels.loc[month] = implied[month]
        levels = levels.sort_index()
    df["IMPLIED_INDEX"] = pd.Series(implied)
    prev = df["IMPLIED_INDEX"].shift(1)
    first = df.index[0] - 1
    if first in cpi.index:
        prev.iloc[0] = cpi.loc[first]
    df["MOM_PCT"] = (df["IMPLIED_INDEX"] / prev - 1) * 100

    stale = df.index[df.index <= LAST_FIXING_MONTH].tolist()
    if stale:
        print(f"Warning: quotes for already-published months {stale}")
    return df


fixing_swaps = get_fixing_swaps()
fixing_swaps[["ticker", "NAME", "MID_BP", "YOY_PCT", "IMPLIED_INDEX", "MOM_PCT", "LAST_UPDATE_DT"]].round(4)

In [ ]:
# Published vs market-implied YoY and MoM (last 3 years of history + fixing swaps)
hist_tbl = cpi_table.loc[LAST_FIXING_MONTH - 35:]
x_hist = hist_tbl.index.to_timestamp()
x_fix = fixing_swaps.index.to_timestamp()

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_hist, y=hist_tbl["YOY_PCT"], mode="lines+markers", name="YoY published"))
fig.add_trace(go.Scatter(x=x_fix, y=fixing_swaps["YOY_PCT"], mode="lines+markers", name=f"YoY implied ({FIXING_SOURCE})",
                         line=dict(dash="dash"), customdata=fixing_swaps[["ticker"]],
                         hovertemplate="%{y:.3f}%<br>%{customdata[0]}<extra></extra>"))
fig.add_trace(go.Bar(x=x_hist, y=hist_tbl["MOM_PCT"], name="MoM published", opacity=0.5, yaxis="y2"))
fig.add_trace(go.Bar(x=x_fix, y=fixing_swaps["MOM_PCT"], name="MoM implied", opacity=0.5, yaxis="y2"))
fig.update_layout(
    title="HICPxT: published vs fixing-swap implied inflation",
    yaxis=dict(title="YoY (%)"),
    yaxis2=dict(title="MoM (%)", overlaying="y", side="right", showgrid=False),
    hovermode="x unified", height=500, barmode="group",
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

### 2.5 ZC inflation caps & floors (ICAP)

ICAP pages *Inflation → Caps & Floors → EURO HICP XT ZC*, two versions of the same grid, in **bp of notional**:

| Version | Page | Ticker | Paid |
|---|---|---|---|
| **Forward** premium | Running/Forward Premium | `EUZP<code><T> ICPI Curncy` | at maturity (undiscounted) |
| **Spot** premium | Spot Premium | `EUIZ<code><T> ICPI Curncy` | upfront (discounted) |

So spot ≈ forward × $P(0,T)$: the ratio spot/forward is ICAP's implied discount factor (compared with our €STR
curve below).

| Floor strike | −1% | −0.5% | 0% | 0.5% | 1% | 1.5% | 2% |
|---|---|---|---|---|---|---|---|
| forward code | `FZ` | `FU` | `F0` | `FT` | `F1` | `FS` | `F2` |
| spot code | `FZ` | `FX` | `F0` | `FV` | `F1` | `FB` | `F2` |

| Cap strike | 2% | 2.5% | 3% | 3.5% | 4% | 4.5% | 5% |
|---|---|---|---|---|---|---|---|
| forward code | `C2` | `CR` | `C3` | `CQ` | `C4` | `CP` | `C5` |
| spot code | `C2` | `CE` | `C3` | `CH` | `C4` | `CL` | `C5` |

Maturities 2, 3, 5, 7, 10, 15, 20, 25, 30Y. Forward tickers zero-pad the maturity for some codes (`EUZPFZ02`,
`EUZPF210`) but not others (`EUZPFU2`, `EUZPF02`); spot tickers never do (`EUIZF22`, `EUIZC210`).
Implied vols are calibrated in 7.2 from the version chosen by `ZC_OPT_PREMIUM_TYPE`.

In [ ]:
ZC_OPT_TENORS = [2, 3, 5, 7, 10, 15, 20, 25, 30]
ZC_OPT_PREMIUM_TYPE = "FWD"   # premiums used for vol calibration (7.1): "FWD" or "SPOT"

ZC_OPT_SPOT_CODES = {   # (kind, strike %) -> ICAP spot-premium code, ticker EUIZ<code><T> ICPI Curncy
    ("floor", -1.0): "FZ", ("floor", -0.5): "FX", ("floor", 0.0): "F0", ("floor", 0.5): "FV",
    ("floor", 1.0): "F1", ("floor", 1.5): "FB", ("floor", 2.0): "F2",
    ("cap", 2.0): "C2", ("cap", 2.5): "CE", ("cap", 3.0): "C3", ("cap", 3.5): "CH",
    ("cap", 4.0): "C4", ("cap", 4.5): "CL", ("cap", 5.0): "C5",
}
ZC_OPT_TICKERS = {   # (kind, strike %) -> forward-premium ticker template
    ("floor", -1.0): "EUZPFZ{T:02d} ICPI Curncy",
    ("floor", -0.5): "EUZPFU{T} ICPI Curncy",
    ("floor", 0.0): "EUZPF0{T} ICPI Curncy",
    ("floor", 0.5): "EUZPFT{T} ICPI Curncy",
    ("floor", 1.0): "EUZPF1{T:02d} ICPI Curncy",
    ("floor", 1.5): "EUZPFS{T} ICPI Curncy",
    ("floor", 2.0): "EUZPF2{T:02d} ICPI Curncy",
    ("cap", 2.0): "EUZPC2{T:02d} ICPI Curncy",
    ("cap", 2.5): "EUZPCR{T} ICPI Curncy",
    ("cap", 3.0): "EUZPC3{T:02d} ICPI Curncy",
    ("cap", 3.5): "EUZPCQ{T} ICPI Curncy",
    ("cap", 4.0): "EUZPC4{T:02d} ICPI Curncy",
    ("cap", 4.5): "EUZPCP{T} ICPI Curncy",
    ("cap", 5.0): "EUZPC5{T:02d} ICPI Curncy",
}


def zc_option_ticker(kind, strike_pct, T, premium_type) -> str:
    if premium_type == "FWD":
        return ZC_OPT_TICKERS[(kind, strike_pct)].format(T=T)
    return f"EUIZ{ZC_OPT_SPOT_CODES[(kind, strike_pct)]}{T} ICPI Curncy"


def get_zc_option_quotes() -> pd.DataFrame:
    """ICAP ZC cap/floor premiums (bp), forward and spot, one row per (type, kind, strike, tenor)."""
    keys = [(pt, kind, k, T) for pt in ["FWD", "SPOT"] for (kind, k) in ZC_OPT_TICKERS for T in ZC_OPT_TENORS]
    tickers = [zc_option_ticker(kind, k, T, pt) for pt, kind, k, T in keys]
    df = bdp(tickers, ["NAME", "PX_BID", "PX_ASK", "PX_LAST", "LAST_UPDATE_DT"],
             numeric=["PX_BID", "PX_ASK", "PX_LAST"]).reset_index()
    df[["premium_type", "kind", "strike_pct", "tenor_y"]] = pd.DataFrame(keys)
    df["PREMIUM_BP"] = df[["PX_BID", "PX_ASK"]].mean(axis=1).fillna(df["PX_LAST"])
    df["LAST_UPDATE_DT"] = pd.to_datetime(df["LAST_UPDATE_DT"], errors="coerce")
    df["STALE"] = df["LAST_UPDATE_DT"] < df["LAST_UPDATE_DT"].max()
    return df


zc_opt_quotes_all = get_zc_option_quotes()
latest = zc_opt_quotes_all["LAST_UPDATE_DT"].max()
for pt, g in zc_opt_quotes_all.groupby("premium_type"):
    print(f"{pt}: {g['PREMIUM_BP'].notna().sum()} / {len(g)} quotes, "
          f"{g['STALE'].sum()} older than {latest:%d-%b-%Y} (mostly deep OTM)")


def premium_grid(kind, premium_type):
    q = zc_opt_quotes_all.query("kind == @kind and premium_type == @premium_type")
    g = q.pivot(index="tenor_y", columns="strike_pct", values="PREMIUM_BP")
    g.columns = [f"{c:.1f}%" for c in g.columns]
    return g


for pt in ["FWD", "SPOT"]:
    print(f"Floors - {pt} premium (bp)")
    display(premium_grid("floor", pt))
    print(f"Caps - {pt} premium (bp)")
    display(premium_grid("cap", pt))

## 3. Inflation curve building (HICPxT)

Goal: a **monthly projected CPI path** $I(m)$ for every reference month, consistent with all market quotes.

**Anchors** (points we treat as known, in priority order):

| Source | Months | How |
|---|---|---|
| Published fixings | up to the last release | `CPTFEMU Index` (2.3) |
| BGC fixing swaps | next ~12 months (`F`), optionally the 2nd year (`H`) | implied index (2.4) |
| ICAP ZC swaps | annual pillars after the fixing-swap window | $I(B+12n) = I(B)\,(1+K_n)^n$ |

$B$ = swap base month = trade month − 3 (no interpolation). No bootstrap is needed: a ZC inflation swap has a
single cashflow, so discounting cancels out and each quote gives a forward CPI directly.

**Between anchors**, the curve is filled with **fixed seasonality factors** $s_1,\dots,s_{12}$ (monthly MoM deviations
from the average, in %, summing to zero):

1. cumulative seasonal $S(m) = \sum_{k \le m} \ln(1 + s_{\text{month}(k)}/100)$, periodic over 12 months;
2. remove it from the anchors: $L^{sa}(m) = \ln I(m) - S(m)$;
3. interpolate $L^{sa}$ **linearly in months** (constant seasonally-adjusted forward inflation);
4. add it back: $I(m) = \exp\left(L^{sa}(m) + S(m)\right)$.

Because the factors sum to zero, seasonality changes the **shape inside each year** but never moves the annual pillars.

### 3.1 Seasonality input

**Example values** to start: rounded from the HICPxT history (2010–2026, excluding 2020–2023), to be replaced by
the desk's own (e.g. Totem). Units: % MoM deviation from the average monthly inflation. If they don't sum to zero,
the mean is removed (with a warning).

In [ ]:
# ---- Seasonality factors (EXAMPLE values - update with desk figures) ---------
SEASONALITY = {
    1: -1.05,   # Jan  (January sales)
    2:  0.30,   # Feb
    3:  0.90,   # Mar  (new clothing collections)
    4:  0.30,   # Apr  (Easter / travel)
    5:  0.00,   # May
    6: -0.05,   # Jun
    7: -0.55,   # Jul  (summer sales)
    8:  0.05,   # Aug
    9:  0.25,   # Sep
    10: 0.05,   # Oct
    11: -0.30,  # Nov
    12: 0.10,   # Dec
}
# -----------------------------------------------------------------------------


def normalise_seasonality(factors: dict) -> pd.Series:
    s = pd.Series(factors, dtype=float).reindex(range(1, 13))
    if s.isna().any():
        raise ValueError(f"Missing seasonality for months {s.index[s.isna()].tolist()}")
    if abs(s.sum()) > 1e-9:
        print(f"Seasonality sums to {s.sum():+.4f}% -> mean removed.")
        s = s - s.mean()
    return s


def historical_seasonality(cpi: pd.Series, exclude_years=(2020, 2021, 2022, 2023)) -> pd.Series:
    """For comparison only: average demeaned log-MoM by calendar month."""
    mom = np.log(cpi).diff().dropna() * 100
    mom = mom[~mom.index.year.isin(exclude_years)]
    by_month = mom.groupby(mom.index.month).mean()
    return by_month - by_month.mean()


seasonality = normalise_seasonality(SEASONALITY)
seas_cmp = pd.DataFrame({"input": seasonality, "historical": historical_seasonality(cpi_fixings)})
seas_cmp.index = pd.to_datetime(seas_cmp.index, format="%m").strftime("%b")

fig = px.bar(seas_cmp, barmode="group", title="Seasonality factors: input vs historical estimate",
             labels={"value": "MoM deviation (%)", "index": "", "variable": ""})
fig.update_layout(height=400)
fig.show()
seas_cmp.round(3).T

### 3.2 Curve construction

Settings:
- `TRADE_DATE` → swap base month $B$ = trade month − 3.
- `USE_H_FIXINGS`: use the 2nd-year BGC contracts too. Default `False` (they looked inconsistent with the 2Y ZC swap
  in 2.4); the ZC swaps then drive the curve after the first year of fixings.
- A ZC pillar falling inside the fixing-swap window is **not** used as an anchor (fixings take priority); it still
  appears in the repricing check (3.3).

In [ ]:
TRADE_DATE = pd.Timestamp.today().normalize()
USE_H_FIXINGS = False


def build_inflation_curve(cpi, fixing_swaps, inf_swaps, seasonality,
                          trade_date=TRADE_DATE, use_h_fixings=USE_H_FIXINGS) -> pd.DataFrame:
    """Monthly HICPxT projection, indexed by reference month."""
    base_month = trade_date.to_period("M") - 3
    if base_month not in cpi.index:
        raise ValueError(f"Base month {base_month} not published yet.")
    base_index = cpi.loc[base_month]

    # 1) Anchors: published > fixing swaps > ZC pillars
    anchors = {m: (v, "published") for m, v in cpi.items()}

    fx = fixing_swaps.dropna(subset=["IMPLIED_INDEX"])
    if not use_h_fixings:
        fx = fx[fx["ticker"].str.contains("EUSWIF")]
    for m, row in fx.iterrows():
        if m not in anchors:
            anchors[m] = (row["IMPLIED_INDEX"], "fixing swap")
    last_fixing_anchor = max(anchors)

    pillars = []
    for tenor, row in inf_swaps.iterrows():
        m = base_month + 12 * int(tenor)
        level = base_index * (1 + row["MID"] / 100) ** tenor
        pillars.append((tenor, m, level))
        if m > last_fixing_anchor:
            anchors[m] = (level, "ZC swap")

    # 2) Monthly grid from the first published month to the last pillar
    grid = pd.period_range(min(anchors), max(anchors), freq="M")
    s_log = np.log1p(seasonality / 100)
    S = pd.Series(s_log.reindex(grid.month).to_numpy(), index=grid).cumsum()

    anchor_s = pd.Series({m: v for m, (v, _) in anchors.items()}).sort_index()
    L_sa = (np.log(anchor_s) - S.reindex(anchor_s.index)).reindex(grid)

    # 3) Linear interpolation of the seasonally-adjusted log index (in months)
    pos = np.arange(len(grid))
    known = L_sa.notna().to_numpy()
    L_sa_full = np.interp(pos, pos[known], L_sa.to_numpy()[known])

    curve = pd.DataFrame(index=grid)
    curve.index.name = "ref_month"
    curve["INDEX"] = np.exp(L_sa_full + S.to_numpy())
    curve["INDEX_SA"] = np.exp(L_sa_full)          # seasonally adjusted (level shifted by S)
    curve["SOURCE"] = [anchors[m][1] if m in anchors else "interpolated" for m in grid]
    curve["MOM_PCT"] = curve["INDEX"].pct_change() * 100
    curve["YOY_PCT"] = curve["INDEX"].pct_change(12) * 100

    curve.attrs.update(base_month=base_month, base_index=base_index, pillars=pillars)
    return curve


inf_curve = build_inflation_curve(cpi_fixings, fixing_swaps, inf_swaps, seasonality)
BASE_MONTH = inf_curve.attrs["base_month"]
print(f"Trade date {TRADE_DATE:%d-%b-%Y} -> base month {BASE_MONTH}, base index {inf_curve.attrs['base_index']:.2f}")
print(inf_curve["SOURCE"].value_counts().to_string())
inf_curve.loc[LAST_FIXING_MONTH - 2: LAST_FIXING_MONTH + 26].round(4)

In [ ]:
def project_cpi(month, curve=None) -> float:
    """Projected (or published) HICPxT for a reference month, e.g. '2030-08' or pd.Period."""
    curve = inf_curve if curve is None else curve
    month = pd.Period(month, freq="M")
    if month not in curve.index:
        raise KeyError(f"{month} outside curve range {curve.index[0]} .. {curve.index[-1]}")
    return float(curve.loc[month, "INDEX"])


print(f"I({BASE_MONTH + 60}) = {project_cpi(BASE_MONTH + 60):.4f}")
print(f"I(2036-08) = {project_cpi('2036-08'):.4f}")

### 3.3 Repricing checks

Each instrument is repriced off the curve; differences should be ~0 for anchors. The 1Y ZC pillar (inside the
fixing-swap window) shows how consistent the BGC fixings are with the ICAP 1Y swap, and the unused `H` fixings show
how far the 2nd-year BGC quotes are from the curve.

In [ ]:
base_index = inf_curve.attrs["base_index"]
zc_check = pd.DataFrame(
    [(t, m, inf_swaps.loc[t, "MID"],
      ((project_cpi(m) / base_index) ** (1 / t) - 1) * 100,
      inf_curve.loc[m, "SOURCE"])
     for t, m, _ in inf_curve.attrs["pillars"]],
    columns=["tenor_y", "ref_month", "market_pct", "curve_pct", "curve_source"],
).set_index("tenor_y")
zc_check["diff_bp"] = (zc_check["curve_pct"] - zc_check["market_pct"]) * 100

fx_check = fixing_swaps[["ticker", "YOY_PCT"]].copy()
fx_check["curve_yoy_pct"] = [inf_curve.loc[m, "YOY_PCT"] if m in inf_curve.index else np.nan for m in fx_check.index]
fx_check["diff_bp"] = (fx_check["curve_yoy_pct"] - fx_check["YOY_PCT"]) * 100
fx_check["used"] = [m in inf_curve.index and inf_curve.loc[m, "SOURCE"] == "fixing swap" for m in fx_check.index]

print("ZC swaps")
display(zc_check.round(4))
print("Fixing swaps")
display(fx_check.round(4))

### 3.4 Charts

In [ ]:
# Projected index by source (last 3y of history + projection)
view = inf_curve.loc[LAST_FIXING_MONTH - 36:].reset_index()
view["date"] = view["ref_month"].dt.to_timestamp()
fig = px.line(view, x="date", y="INDEX", title="HICPxT: published + projected index", height=450)
fig.update_traces(line=dict(color="lightgray"), name="curve", showlegend=True)
for src, sub in view[view["SOURCE"] != "interpolated"].groupby("SOURCE"):
    fig.add_scatter(x=sub["date"], y=sub["INDEX"], mode="markers", name=src)
add_date_vline(fig, BASE_MONTH.to_timestamp(), "swap base")
fig.update_layout(hovermode="x unified", yaxis_title="Index", xaxis_title="")
fig.show()

In [ ]:
# Effect of seasonality: same anchors, flat seasonality vs input factors (first 5 years)
flat = build_inflation_curve(cpi_fixings, fixing_swaps, inf_swaps, seasonality * 0)
win = slice(LAST_FIXING_MONTH - 12, LAST_FIXING_MONTH + 60)
cmp_seas = pd.DataFrame({
    "MoM with seasonality": inf_curve.loc[win, "MOM_PCT"],
    "MoM flat seasonality": flat.loc[win, "MOM_PCT"],
    "YoY with seasonality": inf_curve.loc[win, "YOY_PCT"],
})
cmp_seas.index = cmp_seas.index.to_timestamp()

fig = go.Figure()
fig.add_bar(x=cmp_seas.index, y=cmp_seas["MoM with seasonality"], name="MoM with seasonality")
fig.add_bar(x=cmp_seas.index, y=cmp_seas["MoM flat seasonality"], name="MoM flat seasonality", opacity=0.5)
fig.add_scatter(x=cmp_seas.index, y=cmp_seas["YoY with seasonality"], name="YoY", yaxis="y2", mode="lines")
add_date_vline(fig, LAST_FIXING_MONTH.to_timestamp(), "last fixing")
fig.update_layout(title="Projected MoM / YoY: impact of seasonality", barmode="overlay", height=480,
                  yaxis=dict(title="MoM (%)"),
                  yaxis2=dict(title="YoY (%)", overlaying="y", side="right", showgrid=False),
                  hovermode="x unified", legend=dict(orientation="h", y=-0.15))
fig.show()

## 4. Inflation swap pricer

1. €STR discount curve (needed to PV anything)
2. Classic **zero-coupon** inflation swap: NPV, par rate, IE01 / DV01
3. **Forward-starting** ZC swap: linear formula vs convexity-adjusted, and *why* an adjustment is needed
4. **Year-on-year** swap: a strip of 1-year forward swaps, where convexity really matters

### 4.1 €STR discount curve

`EESWE<n> Curncy` = EUR €STR OIS par rates (annual fixed vs compounded €STR). ICAP pages (`EESWE<n> ICPI Curncy`)
are not populated on this terminal, so the default source is the generic Bloomberg ticker (with the same
ICAP → BBG fallback logic as the inflation swaps).

Bootstrap (simplified, good enough for learning): annual payments, year fractions = 1, valuation = trade date
(no spot lag, no calendar). For an $n$-year par swap: $S_n \sum_{i=1}^{n} P_i = 1 - P_n \Rightarrow
P_n = \dfrac{1 - S_n \sum_{i<n} P_i}{1 + S_n}$. Missing years are filled by linear interpolation of par rates;
discount factors are log-linearly interpolated (flat zero rate beyond 30Y).

In [ ]:
# ---- Source selection ------------------------------------------------------
OIS_SOURCE = "BBG"           # "ICAP" (EESWE<n> ICPI Curncy - empty on this terminal) or "BBG"
FALLBACK_OIS_TO_BBG = True
# -----------------------------------------------------------------------------

OIS_TENORS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 20, 25, 30]
OIS_SOURCES = {"ICAP": "EESWE{n} ICPI Curncy", "BBG": "EESWE{n} Curncy"}


def _ois_snapshot(tenors, source) -> pd.DataFrame:
    tickers = [OIS_SOURCES[source].format(n=t) for t in tenors]
    df = bdp(tickers, ["NAME", "PX_BID", "PX_ASK", "PX_LAST", "LAST_UPDATE_DT"],
             numeric=["PX_BID", "PX_ASK", "PX_LAST"]).reset_index()
    df.insert(0, "tenor_y", list(tenors))
    df["SOURCE"] = source
    return df.set_index("tenor_y")


def get_ois_quotes(source=OIS_SOURCE, fallback=FALLBACK_OIS_TO_BBG) -> pd.DataFrame:
    df = _ois_snapshot(OIS_TENORS, source)
    missing = df.index[df["PX_LAST"].isna()].tolist()
    if fallback and source != "BBG" and missing:
        print(f"{source} missing for tenors {missing} -> using BBG generic tickers.")
        df.loc[missing] = _ois_snapshot(missing, "BBG")
    df["MID"] = df[["PX_BID", "PX_ASK"]].mean(axis=1).fillna(df["PX_LAST"])
    return df


def bootstrap_ois(par_pct: pd.Series) -> pd.Series:
    """Annual €STR OIS par rates (%) -> discount factors at integer years."""
    years = np.arange(1, int(par_pct.index.max()) + 1)
    par = np.interp(years, par_pct.index.astype(float), par_pct.to_numpy(float)) / 100
    dfs, annuity = [], 0.0
    for s in par:
        df = (1 - s * annuity) / (1 + s)
        dfs.append(df)
        annuity += df
    return pd.Series(dfs, index=years, name="DF")


class DiscountCurve:
    """Log-linear interpolation on discount factors, flat zero rate beyond the last pillar."""

    def __init__(self, dfs: pd.Series):
        self.t = np.r_[0.0, dfs.index.to_numpy(float)]
        self.logdf = np.r_[0.0, np.log(dfs.to_numpy(float))]

    def df(self, t):
        t = np.asarray(t, float)
        out = np.interp(t, self.t, self.logdf)
        out = np.where(t > self.t[-1], self.logdf[-1] * t / self.t[-1], out)
        return np.exp(out) if out.ndim else float(np.exp(out))

    def zero(self, t):
        t = np.asarray(t, float)
        return -np.log(self.df(t)) / t


def yf(date, ref=None) -> float:
    """Act/365 year fraction from the trade date."""
    ref = TRADE_DATE if ref is None else ref
    return (pd.Timestamp(date) - ref).days / 365.0


ois_quotes = get_ois_quotes()
ois_curve = DiscountCurve(bootstrap_ois(ois_quotes["MID"]))

ois_view = ois_quotes[["ticker", "SOURCE", "MID"]].copy()
ois_view["DF"] = ois_curve.df(ois_view.index.to_numpy(float))
ois_view["ZERO_PCT"] = ois_curve.zero(ois_view.index.to_numpy(float)) * 100
ois_view.round(5)

In [ ]:
fig = go.Figure()
fig.add_scatter(x=ois_view.index, y=ois_view["MID"], mode="lines+markers", name="€STR OIS par")
fig.add_scatter(x=ois_view.index, y=ois_view["ZERO_PCT"], mode="lines+markers", name="€STR zero (ann. cont.)")
fig.add_scatter(x=inf_swaps.index, y=inf_swaps["MID"], mode="lines+markers", name="HICPxT ZC swap", yaxis="y")
fig.update_layout(title="Nominal (€STR) vs inflation (HICPxT ZC) curves", xaxis_title="Maturity (years)",
                  yaxis_title="%", hovermode="x unified", height=420)
fig.show()

### 4.2 Classic zero-coupon inflation swap

Single exchange at maturity $T$ (years from the trade date), notional $N$, base month $B$ = trade month − 3:

$$\text{Inflation leg} = N\,P(0,T)\left(\frac{I(B+12T)}{I(B)} - 1\right) \qquad
\text{Fixed leg} = N\,P(0,T)\left((1+K)^T - 1\right)$$

- The inflation leg is **linear** in the forward CPI $I(B+12T)$, which is exactly what the curve gives us
  (forward CPI = expectation under the $T$-forward measure). **No convexity** here.
- Par rate: $K^* = \left(\frac{I(B+12T)}{I(B)}\right)^{1/T} - 1$, independent of discounting.
- **IE01**: NPV change for +1 bp on all ZC swap quotes (curve rebuilt). **DV01**: NPV change for +1 bp on all
  €STR quotes. A ZC swap at par has almost no DV01 (both legs share the same discount factor).

In [ ]:
def zc_index_ratio(tenor_y, curve=None) -> float:
    curve = inf_curve if curve is None else curve
    b = curve.attrs["base_month"]
    return project_cpi(b + int(round(12 * tenor_y)), curve) / project_cpi(b, curve)


def price_zc_swap(notional, tenor_y, fixed_pct, receive_inflation=True, infl=None, disc=None) -> dict:
    infl = inf_curve if infl is None else infl
    disc = ois_curve if disc is None else disc
    ratio = zc_index_ratio(tenor_y, infl)
    df = disc.df(tenor_y)
    pv_inf = notional * df * (ratio - 1)
    pv_fix = notional * df * ((1 + fixed_pct / 100) ** tenor_y - 1)
    sign = 1 if receive_inflation else -1
    return {
        "tenor_y": tenor_y, "fixed_pct": fixed_pct, "index_ratio": ratio,
        "par_pct": (ratio ** (1 / tenor_y) - 1) * 100, "df": df,
        "pv_inflation_leg": pv_inf, "pv_fixed_leg": pv_fix, "npv": sign * (pv_inf - pv_fix),
    }


def bumped_inflation_curve(bp=1.0):
    bumped = inf_swaps.copy()
    bumped["MID"] = bumped["MID"] + bp / 100
    return build_inflation_curve(cpi_fixings, fixing_swaps, bumped, seasonality)


def bumped_ois_curve(bp=1.0):
    return DiscountCurve(bootstrap_ois(ois_quotes["MID"] + bp / 100))


INF_CURVE_UP = bumped_inflation_curve(+1)
OIS_CURVE_UP = bumped_ois_curve(+1)


def zc_swap_risk(notional, tenor_y, fixed_pct, receive_inflation=True) -> dict:
    base = price_zc_swap(notional, tenor_y, fixed_pct, receive_inflation)
    base["IE01"] = price_zc_swap(notional, tenor_y, fixed_pct, receive_inflation, infl=INF_CURVE_UP)["npv"] - base["npv"]
    base["DV01"] = price_zc_swap(notional, tenor_y, fixed_pct, receive_inflation, disc=OIS_CURVE_UP)["npv"] - base["npv"]
    return base


# Every market swap repriced at its own quote: NPV ~ 0 (1Y: fixing swaps have priority, see 3.3)
par_check = pd.DataFrame([price_zc_swap(10e6, t, inf_swaps.loc[t, "MID"]) for t in inf_swaps.index]).set_index("tenor_y")
par_check[["fixed_pct", "par_pct", "index_ratio", "df", "pv_inflation_leg", "npv"]].round(4)

In [ ]:
# Example: receive inflation on EUR 10m, 10Y, fixed at 2.00% (below par -> positive NPV)
example = zc_swap_risk(10e6, 10, 2.00, receive_inflation=True)
pd.Series(example).to_frame("10Y receive inflation @ 2.00%").style.format("{:,.4f}")

### 4.3 Forward-starting ZC swap: linear formula vs convexity

A forward ZC swap from $T_1$ to $T_2$ pays at $T_2$: $N\left(\frac{I(T_2)}{I(T_1)} - 1\right)$ vs
$N\left((1+K)^{T_2-T_1} - 1\right)$. The base index $I(T_1)$ is **unknown today**.

**Linear formula** (what you'd naively do with the curve): replace each index by its forward,
$$\frac{I(T_2)}{I(T_1)} \approx \frac{\hat I_2(0)}{\hat I_1(0)}, \qquad \hat I_i(0) = E^{T_i}\left[I(T_i)\right].$$

**Why it's wrong**: the payoff is a *ratio* of two random quantities paid at $T_2$, so its value is
$E^{T_2}\left[\frac{I(T_2)}{I(T_1)}\right] \neq \frac{E^{T_2}[I(T_2)]}{E^{T_1}[I(T_1)]}$:
1. **Jensen / ratio effect**: $E[1/X] > 1/E[X]$. If $I(T_1)$ and the forward $\hat I_2$ moved perfectly together
   (same vol, correlation 1) the ratio would be deterministic → no adjustment. Decorrelation between forward indices
   (i.e. real-rate vol) creates it.
2. **Measure effect**: $\hat I_1$ is a martingale under the $T_1$-forward measure but the payoff is paid at $T_2$.
   Changing measure introduces the covariance between inflation and **nominal rates** over $[T_1, T_2]$.

**Mercurio (2005) market model** (lognormal forward CPIs, vols $\sigma_{I,1}, \sigma_{I,2}$, correlation
$\rho_{II}$, nominal forward rate $F$ over $[T_1,T_2]$ with normal vol $\sigma_N$, inflation/nominal correlation $\rho_{In}$):
$$E^{T_2}\left[\frac{I(T_2)}{I(T_1)}\right] = \frac{\hat I_2(0)}{\hat I_1(0)}\,e^{D}, \qquad
D = \sigma_{I,1}T_1\left(\sigma_{I,1} - \rho_{II}\,\sigma_{I,2} + \rho_{In}\,\frac{\sigma_N\,\tau}{1+\tau F}\right),\; \tau = T_2 - T_1$$

The parameters below are **example inputs** (to be calibrated later, e.g. to YoY vs ZC caps).

In [ ]:
# ---- Convexity model inputs (EXAMPLE values) ---------------------------------
CONVEXITY_PARAMS = dict(
    sigma_I=0.015,    # lognormal vol of forward CPIs (1.5%)
    rho_II=0.90,      # correlation between consecutive forward CPIs
    rho_In=0.30,      # correlation forward CPI / nominal forward rate
    sigma_N=0.0080,   # normal vol of nominal forward rates (80 bp)
)
# -----------------------------------------------------------------------------


def mercurio_D(T1, T2, sigma_I, rho_II, rho_In, sigma_N, sigma_I2=None, disc=None) -> float:
    disc = ois_curve if disc is None else disc
    sigma_I2 = sigma_I if sigma_I2 is None else sigma_I2
    tau = T2 - T1
    F = (disc.df(T1) / disc.df(T2) - 1) / tau
    return sigma_I * T1 * (sigma_I - rho_II * sigma_I2 + rho_In * sigma_N * tau / (1 + tau * F))


def forward_zc(T1, T2, params=CONVEXITY_PARAMS, curve=None) -> dict:
    curve = inf_curve if curve is None else curve
    b = curve.attrs["base_month"]
    ratio_lin = project_cpi(b + 12 * T2, curve) / project_cpi(b + 12 * T1, curve)
    D = mercurio_D(T1, T2, **params)
    ratio_adj = ratio_lin * np.exp(D)
    tau = T2 - T1
    lin = (ratio_lin ** (1 / tau) - 1) * 100
    adj = (ratio_adj ** (1 / tau) - 1) * 100
    return {"fwd": f"{T1}y{tau}y", "T1": T1, "T2": T2, "linear_pct": lin, "adjusted_pct": adj,
            "convexity_bp": (adj - lin) * 100, "D": D,
            "pv_diff_per_10m": 10e6 * ois_curve.df(T2) * (ratio_adj - ratio_lin)}


fwd_table = pd.DataFrame([forward_zc(a, b) for a, b in [(1, 2), (2, 3), (5, 6), (5, 10), (9, 10), (10, 20), (20, 30)]]).set_index("fwd")
fwd_table.round(4)

**Monte Carlo check** of the ratio effect (deterministic nominal rates → the $\rho_{In}$ term drops):
simulate $\hat I_1(T_1) = I(T_1)$ and $\hat I_2(T_1)$ as correlated lognormal martingales and compare $E[\hat I_2/\hat I_1]$
with the linear ratio and with the closed form $e^{\sigma_1 T_1(\sigma_1 - \rho\,\sigma_2)}$.
With $\rho = 1$ and $\sigma_1 = \sigma_2$ the adjustment vanishes.

In [ ]:
def mc_forward_ratio(T1, I1, I2, s1, s2, rho, n=1_000_000, seed=42) -> float:
    rng = np.random.default_rng(seed)
    z1 = rng.standard_normal(n)
    z2 = rho * z1 + np.sqrt(1 - rho ** 2) * rng.standard_normal(n)
    I1_T = I1 * np.exp(-0.5 * s1 ** 2 * T1 + s1 * np.sqrt(T1) * z1)
    I2_T = I2 * np.exp(-0.5 * s2 ** 2 * T1 + s2 * np.sqrt(T1) * z2)
    return float(np.mean(I2_T / I1_T))


T1, T2 = 5, 10
I1, I2 = project_cpi(BASE_MONTH + 12 * T1), project_cpi(BASE_MONTH + 12 * T2)
rows = []
for s1, s2, rho in [(0.015, 0.015, 1.0), (0.015, 0.015, 0.9), (0.015, 0.015, 0.5), (0.03, 0.03, 0.9), (0.015, 0.03, 0.9)]:
    mc = mc_forward_ratio(T1, I1, I2, s1, s2, rho)
    closed = I2 / I1 * np.exp(s1 * T1 * (s1 - rho * s2))
    rows.append({"sigma_1": s1, "sigma_2": s2, "rho": rho, "linear": I2 / I1, "closed_form": closed, "monte_carlo": mc,
                 "adj_bp_on_ratio": (closed / (I2 / I1) - 1) * 1e4})
pd.DataFrame(rows).round(6)

In [ ]:
# Convexity on a 1-year forward (YoY-type) as a function of its start date, for several rho_In
starts = np.arange(1, 30)
fig = go.Figure()
for rho_in in [-0.3, 0.0, 0.3, 0.6]:
    prm = {**CONVEXITY_PARAMS, "rho_In": rho_in}
    adj = [forward_zc(int(t), int(t) + 1, prm)["convexity_bp"] for t in starts]
    fig.add_scatter(x=starts, y=adj, mode="lines", name=f"rho_In = {rho_in}")
fig.update_layout(title="Convexity adjustment on 1y forward inflation (bp) vs start T1",
                  xaxis_title="T1 (years)", yaxis_title="adjustment (bp)", hovermode="x unified", height=420)
fig.show()

### 4.4 Year-on-year inflation swap

Each year $i = 1..n$ the inflation leg pays $N\left(\frac{I(T_i)}{I(T_{i-1})} - 1\right)$ against $N K$.
It's a strip of 1-year **forward** swaps, so every period except the first needs the convexity adjustment:
$$K^{YoY} = \frac{\sum_i P(0,T_i)\left(\frac{\hat I_i}{\hat I_{i-1}}e^{D_i} - 1\right)}{\sum_i P(0,T_i)}$$
The linear YoY par rate is a discount-weighted average of the 1y forwards; the convexity adds up period after period.

In [ ]:
def yoy_forwards(n_years, params=CONVEXITY_PARAMS, curve=None, disc=None) -> pd.DataFrame:
    curve = inf_curve if curve is None else curve
    disc = ois_curve if disc is None else disc
    b = curve.attrs["base_month"]
    rows = []
    for i in range(1, n_years + 1):
        ratio = project_cpi(b + 12 * i, curve) / project_cpi(b + 12 * (i - 1), curve)
        D = mercurio_D(i - 1, i, disc=disc, **params)
        rows.append({"period": i, "T_fix": i, "df": disc.df(i), "yoy_linear_pct": (ratio - 1) * 100,
                     "yoy_adjusted_pct": (ratio * np.exp(D) - 1) * 100, "D": D})
    df = pd.DataFrame(rows).set_index("period")
    df["convexity_bp"] = (df["yoy_adjusted_pct"] - df["yoy_linear_pct"]) * 100
    return df


def yoy_swap_par(n_years, params=CONVEXITY_PARAMS) -> dict:
    f = yoy_forwards(n_years, params)
    w = f["df"] / f["df"].sum()
    return {"tenor_y": n_years, "zc_pct": inf_swaps["MID"].get(n_years, np.nan),
            "yoy_linear_pct": (w * f["yoy_linear_pct"]).sum(),
            "yoy_adjusted_pct": (w * f["yoy_adjusted_pct"]).sum()}


yoy_table = pd.DataFrame([yoy_swap_par(n) for n in [1, 2, 3, 5, 7, 10, 15, 20, 30]]).set_index("tenor_y")
yoy_table["convexity_bp"] = (yoy_table["yoy_adjusted_pct"] - yoy_table["yoy_linear_pct"]) * 100
yoy_table.round(4)

In [ ]:
f30 = yoy_forwards(30)
fig = go.Figure()
fig.add_scatter(x=f30.index, y=f30["yoy_linear_pct"], mode="lines+markers", name="1y fwd YoY - linear")
fig.add_scatter(x=f30.index, y=f30["yoy_adjusted_pct"], mode="lines+markers", name="1y fwd YoY - convexity adjusted")
fig.add_bar(x=f30.index, y=f30["convexity_bp"], name="adjustment (bp)", yaxis="y2", opacity=0.4)
fig.update_layout(title="Forward YoY inflation: linear vs convexity-adjusted", xaxis_title="period (years)",
                  yaxis=dict(title="YoY (%)"), yaxis2=dict(title="bp", overlaying="y", side="right", showgrid=False),
                  hovermode="x unified", height=450, legend=dict(orientation="h", y=-0.2))
fig.show()

## 5. Linker pricer: OAT€i 0.1% 25-Jul-2036

**How a linker quoted in real terms works**

- The bond pays **fixed real cash flows**: coupon $c$ on a notional of 100 every year, 100 at maturity.
- Each nominal payment = real cash flow × **index ratio** $IR(d) = \frac{\text{Ref}(d)}{\text{Base}}$ on the payment date.
- **Daily reference index** (OAT€i, 3-month lag, linear interpolation inside the month):
  $\text{Ref}(d) = I(m-3) + \frac{\text{day}(d)-1}{\text{days in } m}\left(I(m-2) - I(m-3)\right)$, $m$ = month of $d$.
- **Base** = the reference index at the bond's base date (`BASE_CPI`, expressed in the current HICPxT base).
- Redemption is **floored at par**: $100 \times \max(IR(T), 1)$ (an embedded deflation floor, see 7.5).
- The market quotes a **real clean price** and a **real yield** $y$: the yield discounts the *real* flows only.
  Inflation never appears in the price/yield relation.
- The cash you pay (**invoice**) = (real clean price + real accrued) × $IR(\text{settlement})$.
- Projected nominal flows use the inflation curve (section 3); discounting them on €STR and comparing with the invoice
  gives the linker's **z-spread to €STR**. Comparing with a nominal OAT gives the **breakeven**.

### 5.1 Static data and market quotes

In [ ]:
LINKER_TICKER = "FRTR 0.1 07/25/36 Govt"    # OAT€i 0.1% 25-Jul-2036 - desk label OATe_Jul-36_0.1
NOMINAL_TICKER = "FRTR 1.25 05/25/36 Govt"  # matched nominal OAT (desk pair OAT_May-36_1.25)

LINKER_FIELDS = ["NAME", "ID_ISIN", "CPN", "CPN_FREQ", "MATURITY", "ISSUE_DT", "REFERENCE_INDEX", "BASE_CPI",
                 "DAY_CNT_DES", "SETTLE_DT", "PREV_CPN_DT", "NXT_CPN_DT", "IDX_RATIO", "INT_ACC",
                 "PX_BID", "PX_ASK", "PX_LAST", "YLD_YTM_MID", "AMT_OUTSTANDING"]
LINKER_NUMERIC = ["CPN", "CPN_FREQ", "BASE_CPI", "IDX_RATIO", "INT_ACC", "PX_BID", "PX_ASK", "PX_LAST",
                  "YLD_YTM_MID", "AMT_OUTSTANDING"]

lk_raw = bdp([LINKER_TICKER], LINKER_FIELDS, numeric=LINKER_NUMERIC).iloc[0]
lk = {
    "cpn": lk_raw["CPN"],
    "maturity": pd.Timestamp(lk_raw["MATURITY"]),
    "base_cpi": lk_raw["BASE_CPI"],
    "settle": pd.Timestamp(lk_raw["SETTLE_DT"]),
    "clean_real": np.nanmean([lk_raw["PX_BID"], lk_raw["PX_ASK"]]),
}
display(lk_raw.to_frame(LINKER_TICKER))
print(f"Mid real clean price: {lk['clean_real']:.4f}")

### 5.2 Daily reference index and index ratio

Checked against Bloomberg's `IDX_RATIO` at the settlement date. Ref index and ratio are rounded to 5 decimals
(AFT convention). Months not yet published come from the projected curve (section 3).

In [ ]:
def ref_cpi(date, curve=None, decimals=5) -> float:
    d = pd.Timestamp(date)
    m = d.to_period("M")
    i3, i2 = project_cpi(m - 3, curve), project_cpi(m - 2, curve)
    return round(i3 + (d.day - 1) / d.days_in_month * (i2 - i3), decimals)


def index_ratio(date, base_cpi, curve=None) -> float:
    return round(ref_cpi(date, curve) / base_cpi, 5)


def ref_cpi_detail(date, curve=None) -> dict:
    d = pd.Timestamp(date)
    m = d.to_period("M")
    c = inf_curve if curve is None else curve
    return {"date": d.date(), "month_m-3": str(m - 3), "I(m-3)": project_cpi(m - 3, c), "src_m-3": c.loc[m - 3, "SOURCE"],
            "month_m-2": str(m - 2), "I(m-2)": project_cpi(m - 2, c), "src_m-2": c.loc[m - 2, "SOURCE"],
            "weight": (d.day - 1) / d.days_in_month, "ref_cpi": ref_cpi(d, c)}


settle_detail = ref_cpi_detail(lk["settle"])
lk["ir_settle"] = index_ratio(lk["settle"], lk["base_cpi"])
settle_detail.update({"base_cpi": lk["base_cpi"], "IR model": lk["ir_settle"], "IR Bloomberg": lk_raw["IDX_RATIO"],
                      "diff": lk["ir_settle"] - lk_raw["IDX_RATIO"]})
pd.Series(settle_detail).to_frame("settlement")

### 5.3 Real price ↔ real yield

Act/Act ICMA, annual coupon. With $v$ = (next coupon − settlement) / (next coupon − previous coupon):
$$P^{dirty}_{real} = \sum_{i=0}^{n-1} \frac{CF^{real}_i}{(1+y)^{v+i}}, \qquad
\text{Accrued}_{real} = c \times \frac{\text{settle} - \text{prev}}{\text{next} - \text{prev}}, \qquad
P^{clean}_{real} = P^{dirty}_{real} - \text{Accrued}_{real}$$

In [ ]:
from scipy.optimize import brentq


def coupon_schedule(maturity, settle, freq=1):
    """Future coupon dates (> settle) and the previous coupon date."""
    step = pd.DateOffset(months=12 // freq)
    dates, d = [], pd.Timestamp(maturity)
    while d > settle:
        dates.append(d)
        d = d - step
    return sorted(dates), d


def real_cashflows(cpn, maturity, settle):
    dates, prev = coupon_schedule(maturity, settle)
    cfs = np.full(len(dates), cpn, dtype=float)
    cfs[-1] += 100
    v = (dates[0] - settle).days / (dates[0] - prev).days
    times = v + np.arange(len(dates))
    accrued = cpn * (settle - prev).days / (dates[0] - prev).days
    return dates, prev, cfs, times, accrued


def real_dirty_from_yield(y, cpn, maturity, settle) -> float:
    _, _, cfs, times, _ = real_cashflows(cpn, maturity, settle)
    return float(np.sum(cfs / (1 + y) ** times))


def real_yield_from_clean(clean, cpn, maturity, settle) -> float:
    *_, accrued = real_cashflows(cpn, maturity, settle)
    return brentq(lambda y: real_dirty_from_yield(y, cpn, maturity, settle) - (clean + accrued), -0.5, 0.5)


dates, prev_cpn, real_cfs, times, accrued_real = real_cashflows(lk["cpn"], lk["maturity"], lk["settle"])
lk["accrued_real"] = accrued_real
lk["dirty_real"] = lk["clean_real"] + accrued_real
lk["real_yield"] = real_yield_from_clean(lk["clean_real"], lk["cpn"], lk["maturity"], lk["settle"])

p_up = real_dirty_from_yield(lk["real_yield"] + 1e-4, lk["cpn"], lk["maturity"], lk["settle"])
p_dn = real_dirty_from_yield(lk["real_yield"] - 1e-4, lk["cpn"], lk["maturity"], lk["settle"])
lk["real_dv01"] = (p_dn - p_up) / 2                       # real price change per 1bp of real yield
lk["real_mod_duration"] = lk["real_dv01"] / lk["dirty_real"] * 1e4

pd.Series({
    "real clean (mid)": lk["clean_real"], "real accrued (model)": accrued_real, "real accrued (BBG INT_ACC)": lk_raw["INT_ACC"],
    "real dirty": lk["dirty_real"], "real yield model (%)": lk["real_yield"] * 100,
    "real yield BBG YLD_YTM_MID (%)": lk_raw["YLD_YTM_MID"], "diff (bp)": (lk["real_yield"] * 100 - lk_raw["YLD_YTM_MID"]) * 100,
    "real DV01 (per 100 real)": lk["real_dv01"], "real modified duration": lk["real_mod_duration"],
}).to_frame("OAT€i 2036").round(6)

### 5.4 Full cash-flow table (every flow recomputed)

For each payment date: reference months, projected CPI, daily reference index, index ratio, real flow, nominal
flow (with par floor at maturity), real-yield discounting (must sum to the real dirty price), and €STR discounting
of the projected nominal flows.

In [ ]:
def linker_cashflow_table(bond, curve=None, disc=None) -> pd.DataFrame:
    curve = inf_curve if curve is None else curve
    disc = ois_curve if disc is None else disc
    dates, prev, cfs, times, _ = real_cashflows(bond["cpn"], bond["maturity"], bond["settle"])
    df_settle = disc.df(yf(bond["settle"]))
    rows = []
    for d, cf, t in zip(dates, cfs, times):
        det = ref_cpi_detail(d, curve)
        ir = round(det["ref_cpi"] / bond["base_cpi"], 5)
        is_mat = d == bond["maturity"]
        coupon_nom = bond["cpn"] * ir
        redemption_nom = 100 * max(ir, 1.0) if is_mat else 0.0
        nominal_cf = coupon_nom + redemption_nom
        df_ois = disc.df(yf(d)) / df_settle
        rows.append({
            "pay_date": d.date(), "t_actact": t, "ref_m-3": det["month_m-3"], "I(m-3)": det["I(m-3)"],
            "ref_m-2": det["month_m-2"], "I(m-2)": det["I(m-2)"], "cpi_source": det["src_m-2"],
            "ref_cpi": det["ref_cpi"], "index_ratio": ir, "real_cf": cf,
            "real_df": (1 + bond["real_yield"]) ** -t, "pv_real": cf * (1 + bond["real_yield"]) ** -t,
            "nominal_coupon": coupon_nom, "nominal_redemption": redemption_nom,
            "floor_active": is_mat and ir < 1, "nominal_cf": nominal_cf,
            "t_settle": (d - bond["settle"]).days / 365.0, "ois_df": df_ois, "pv_nominal_ois": nominal_cf * df_ois,
        })
    return pd.DataFrame(rows).set_index("pay_date")


cf_table = linker_cashflow_table(lk)
cf_table.round(6)

In [ ]:
lk["invoice"] = lk["dirty_real"] * lk["ir_settle"]          # cash paid per 100 nominal
pv_ois = cf_table["pv_nominal_ois"].sum()


def zspread(table, target) -> float:
    f = lambda z: float(np.sum(table["nominal_cf"] * table["ois_df"] * np.exp(-z * table["t_settle"]))) - target
    return brentq(f, -0.5, 0.5)


lk["z_spread_bp"] = zspread(cf_table, lk["invoice"]) * 1e4

pd.Series({
    "sum PV real flows @ real yield": cf_table["pv_real"].sum(),
    "real dirty price (market)": lk["dirty_real"],
    "index ratio @ settlement": lk["ir_settle"],
    "invoice = real dirty x IR (per 100)": lk["invoice"],
    "PV projected nominal flows @ €STR": pv_ois,
    "model - market (per 100)": pv_ois - lk["invoice"],
    "z-spread to €STR + infl. curve (bp)": lk["z_spread_bp"],
}).to_frame("OAT€i 2036").round(6)

### 5.5 Breakeven inflation

- **Bond breakeven** vs the matched nominal OAT: $BE = \frac{1+y_{nom}}{1+y_{real}} - 1$ (Fisher), $\approx y_{nom} - y_{real}$.
- **Swap breakeven** to the linker's maturity, from the inflation curve with the same daily reference index:
  $\left(\frac{\text{Ref}(T)}{\text{Ref}(\text{settle})}\right)^{1/T} - 1$.
- The difference is the **bond/swap inflation basis** for this line.

In [ ]:
nom = bdp([NOMINAL_TICKER], ["NAME", "CPN", "MATURITY", "PX_LAST", "YLD_YTM_MID"],
          numeric=["CPN", "PX_LAST", "YLD_YTM_MID"]).iloc[0]
y_nom = nom["YLD_YTM_MID"] / 100
y_real = lk["real_yield"]
T_lk = (lk["maturity"] - lk["settle"]).days / 365.0
swap_be = (ref_cpi(lk["maturity"]) / ref_cpi(lk["settle"])) ** (1 / T_lk) - 1

pd.Series({
    "nominal OAT": f"{NOMINAL_TICKER} (mat {nom['MATURITY']})",
    "nominal yield (%)": y_nom * 100,
    "real yield (%)": y_real * 100,
    "breakeven simple (%)": (y_nom - y_real) * 100,
    "breakeven Fisher (%)": ((1 + y_nom) / (1 + y_real) - 1) * 100,
    "swap breakeven to Jul-36 (%)": swap_be * 100,
    "bond - swap breakeven (bp)": (((1 + y_nom) / (1 + y_real) - 1) - swap_be) * 1e4,
}).to_frame("OAT€i 2036")

In [ ]:
# Projected index ratio path to maturity + nominal flows
path_dates = pd.date_range(lk["settle"], lk["maturity"], freq="MS").union([lk["settle"], lk["maturity"]])
ir_path = pd.Series([index_ratio(d, lk["base_cpi"]) for d in path_dates], index=path_dates)

fig = go.Figure()
fig.add_scatter(x=ir_path.index, y=ir_path, mode="lines", name="index ratio (projected)")
fig.add_scatter(x=pd.to_datetime(cf_table.index), y=cf_table["index_ratio"], mode="markers", name="IR at payment dates",
                customdata=cf_table[["nominal_cf", "real_cf"]],
                hovertemplate="IR %{y:.5f}<br>nominal CF %{customdata[0]:.4f}<br>real CF %{customdata[1]:.2f}<extra></extra>")
fig.add_hline(y=1.0, line_dash="dot", annotation_text="par floor (IR = 1)")
fig.update_layout(title="OAT€i 2036: projected index ratio", yaxis_title="index ratio", height=420, hovermode="x unified")
fig.show()

## 6. Asset swap (ASW) pricer

In [ ]:
# TODO

## 7. Inflation options: caps & floors

**ZC cap/floor** on HICPxT, notional $N$, maturity $T$, strike $K$, payoff at $T$ on the index ratio
$X = \frac{I(B+12T)}{I(B)}$ (same 3-month lag / base month as the ZC swap):

$$\text{cap} = N\max\left(X - (1+K)^T, 0\right) \qquad \text{floor} = N\max\left((1+K)^T - X, 0\right)$$

Under the $T$-forward measure $E^T[X] = F = (1+K^*_T)^T$ (the ZC swap forward from section 3), so
**cap − floor = ZC swap** whatever the model.

Plan:
- **7.1** two models: **Bachelier** (normal) and **Black** (lognormal)
- **7.2** calibrate each model's vol surface to the **ICAP broker premiums** (2.5)
- **7.3** reprice caps/floors with the implied vols (market strikes, off-grid strikes, parity, greeks)
- **7.4** payoff vs price profiles, **7.5** greeks (delta, gamma, vega) vs inflation level and time
- **7.6** the linker's embedded deflation floor, **7.7** YoY caps/floors

### 7.1 Models

Both price the forward premium $\Pi = E^T[\text{payoff}]$ (per unit notional); spot premium $= P(0,T)\,\Pi$.
With $\bar K = (1+K)^T$:

**Bachelier (normal)**: $X_T \sim \mathcal N(F, s^2)$, $\;d = \frac{F - \bar K}{s}$
$$\text{cap} = (F - \bar K)\,\Phi(d) + s\,\varphi(d) \qquad \text{floor} = (\bar K - F)\,\Phi(-d) + s\,\varphi(d)$$
Vol quoted as a **normal vol of the ZC rate** $\sigma_N$ (bp/yr), mapped to the index ratio with
$\frac{dX}{dr} = T(1+r)^{T-1}$: $\;s = \sigma_N\,T\,(1+F_r)^{T-1}\sqrt T$, $\;F_r = F^{1/T} - 1$.

**Black (lognormal)**: $\ln X_T \sim \mathcal N(\ln F - \tfrac12 \sigma^2 T,\ \sigma^2 T)$
$$\text{cap} = F\,\Phi(d_1) - \bar K\,\Phi(d_2) \qquad \text{floor} = \bar K\,\Phi(-d_2) - F\,\Phi(-d_1), \qquad
d_{1,2} = \frac{\ln(F/\bar K) \pm \frac12\sigma^2 T}{\sigma\sqrt T}$$
Vol $\sigma$ = **lognormal vol of the index ratio** (%/yr).

Same ATM price ⇒ different smiles: Black's distribution is skewed (no negative index, fatter right tail), so for the
same ATM vol it prices low-strike floors cheaper and high-strike caps dearer than Bachelier. Calibrating both to the
same premiums gives two different vol surfaces that reprice the market identically.

In [ ]:
from scipy.optimize import brentq
from scipy.stats import norm
from plotly.subplots import make_subplots

ZC_MODELS = ["bachelier", "black"]
VOL_UNITS = {"bachelier": ("bp", 1e4), "black": ("%", 1e2)}    # display unit, scale


def bachelier(F, K, sd, is_call) -> float:
    """Undiscounted Bachelier price, sd = total standard deviation of the underlying at expiry."""
    if sd <= 0:
        return max(F - K, 0.0) if is_call else max(K - F, 0.0)
    d = (F - K) / sd
    return (F - K) * norm.cdf(d) + sd * norm.pdf(d) if is_call else (K - F) * norm.cdf(-d) + sd * norm.pdf(d)


def black(F, K, sd, is_call) -> float:
    """Undiscounted Black price, sd = sigma * sqrt(T)."""
    if sd <= 0:
        return max(F - K, 0.0) if is_call else max(K - F, 0.0)
    d1 = (np.log(F / K) + 0.5 * sd ** 2) / sd
    d2 = d1 - sd
    return F * norm.cdf(d1) - K * norm.cdf(d2) if is_call else K * norm.cdf(-d2) - F * norm.cdf(-d1)


def zc_fwd_premium(F_ratio, K_ratio, T, vol, model, is_cap) -> float:
    """ZC cap/floor forward premium per unit notional.
    bachelier: vol = normal vol of the ZC rate (decimal, 0.006 = 60bp)
    black    : vol = lognormal vol of the index ratio (decimal, 0.03 = 3%)"""
    if model == "bachelier":
        F_r = F_ratio ** (1 / T) - 1
        return bachelier(F_ratio, K_ratio, vol * T * (1 + F_r) ** (T - 1) * np.sqrt(T), is_cap)
    if model == "black":
        return black(F_ratio, K_ratio, vol * np.sqrt(T), is_cap)
    raise ValueError(model)


def implied_vol(fwd_premium, F_ratio, K_ratio, T, model, is_cap) -> float:
    """Model vol (decimal) reproducing a forward premium per unit notional; NaN if not attainable."""
    lo, hi = (1e-7, 0.05) if model == "bachelier" else (1e-6, 3.0)
    f = lambda v: zc_fwd_premium(F_ratio, K_ratio, T, v, model, is_cap) - fwd_premium
    if not np.isfinite(fwd_premium) or f(lo) >= 0 or f(hi) <= 0:
        return np.nan   # below intrinsic value (stale / inconsistent quote) or out of range
    return brentq(f, lo, hi, xtol=1e-14)


# Illustration: 10Y, Black vol chosen to match the Bachelier ATM price -> compare the smiles they imply
T_ill = 10
F_ill = zc_index_ratio(T_ill)
F_r_ill = F_ill ** (1 / T_ill) - 1
sig_n = 0.0040
atm = zc_fwd_premium(F_ill, F_ill, T_ill, sig_n, "bachelier", True)
sig_ln = implied_vol(atm, F_ill, F_ill, T_ill, "black", True)
print(f"10Y forward ZC {F_r_ill*100:.3f}% | Bachelier {sig_n*1e4:.0f}bp  <=>  Black {sig_ln*100:.3f}% (same ATM premium)")

ks = np.round(np.arange(-1.0, 5.01, 0.25), 2)
fig = go.Figure()
for model, vol in [("bachelier", sig_n), ("black", sig_ln)]:
    for kind in ["cap", "floor"]:
        prem = [zc_fwd_premium(F_ill, (1 + k / 100) ** T_ill, T_ill, vol, model, kind == "cap") * 1e4 for k in ks]
        fig.add_scatter(x=ks, y=prem, mode="lines", name=f"{kind} - {model}",
                        line=dict(dash=None if model == "bachelier" else "dash"))
fig.add_vline(x=F_r_ill * 100, line_dash="dot")
fig.update_layout(title="10Y ZC cap/floor forward premium: Bachelier vs Black, same ATM price",
                  xaxis_title="strike (%)", yaxis_title="forward premium (bp)", yaxis_type="log",
                  hovermode="x unified", height=450)
fig.show()

### 7.2 Vol calibration from broker premiums (ICAP)

1. **Spot vs forward**: the ratio of the two ICAP grids is ICAP's discount factor, compared with our €STR curve.
2. **Calibration set**: the version chosen by `ZC_OPT_PREMIUM_TYPE` (2.5). Spot premiums are converted to forward
   premiums with *our* €STR discount factor, so the vols only depend on option value, not on discounting.
3. For each quote: forward $F$ from **our** inflation curve, then solve the vol that reproduces the premium,
   **separately for each model**.
4. **Surface** per model: OTM side for each strike (floors ≤ 2%, caps ≥ 2%, average of both at 2%), gaps filled
   along strikes.
5. **Parity check at 2%**: ICAP cap − floor vs our swap forward premium $F - 1.02^T$. A gap means ICAP's forward
   differs from ours; it shows up as a cap/floor vol difference at 2% (averaged in the surface).

In [ ]:
# 1) ICAP implied discount factors
sp = zc_opt_quotes_all.pivot_table(index=["kind", "strike_pct", "tenor_y"], columns="premium_type", values="PREMIUM_BP")
sp["icap_df"] = sp["SPOT"] / sp["FWD"]
df_check = sp[sp["FWD"] > 20].groupby("tenor_y")["icap_df"].median().to_frame("ICAP DF (median spot/fwd, prem > 20bp)")
df_check["€STR DF"] = [ois_curve.df(T) for T in df_check.index]
df_check["diff"] = df_check.iloc[:, 0] - df_check["€STR DF"]
print("Spot vs forward premiums -> ICAP implied discount factors")
display(df_check.round(4))

# 2-3) Implied vols, both models
opt = zc_opt_quotes_all.query("premium_type == @ZC_OPT_PREMIUM_TYPE").copy()
opt["df"] = opt["tenor_y"].map(lambda T: ois_curve.df(T))
opt["MKT_FWD_BP"] = opt["PREMIUM_BP"] / opt["df"] if ZC_OPT_PREMIUM_TYPE == "SPOT" else opt["PREMIUM_BP"]
opt["fwd_ratio"] = [zc_index_ratio(T) for T in opt["tenor_y"]]
opt["fwd_zc_pct"] = (opt["fwd_ratio"] ** (1 / opt["tenor_y"]) - 1) * 100
opt["K_ratio"] = (1 + opt["strike_pct"] / 100) ** opt["tenor_y"]
for model in ZC_MODELS:
    unit, scale = VOL_UNITS[model]
    opt[f"vol_{model}"] = [implied_vol(r.MKT_FWD_BP / 1e4, r.fwd_ratio, r.K_ratio, r.tenor_y, model, r.kind == "cap") * scale
                           for r in opt.itertuples()]

failed = opt[opt[[f"vol_{m}" for m in ZC_MODELS]].isna().any(axis=1)]
print(f"Calibrated on {ZC_OPT_PREMIUM_TYPE} premiums: {len(opt) - len(failed)} / {len(opt)} quotes")
if not failed.empty:
    print("No implied vol (premium below intrinsic vs our forward, usually stale):")
    display(failed[["ticker", "PREMIUM_BP", "LAST_UPDATE_DT", "fwd_zc_pct"]])

# 4) Surfaces (vol in display units: bp for Bachelier, % for Black)
SURFACES = {}
for model in ZC_MODELS:
    s = opt.groupby(["tenor_y", "strike_pct"])[f"vol_{model}"].mean().unstack("strike_pct")
    SURFACES[model] = s.interpolate(axis=1, limit_direction="both").ffill().bfill()
    print(f"{model} implied vol surface ({VOL_UNITS[model][0]})")
    display(SURFACES[model].round(2))

In [ ]:
# 5) Put-call parity at 2%
c2 = opt.query("kind == 'cap' and strike_pct == 2.0").set_index("tenor_y")
f2 = opt.query("kind == 'floor' and strike_pct == 2.0").set_index("tenor_y")
parity_mkt = pd.DataFrame({
    "ICAP cap-floor 2% (fwd bp)": c2["MKT_FWD_BP"] - f2["MKT_FWD_BP"],
    "our swap F - 1.02^T (fwd bp)": (c2["fwd_ratio"] - 1.02 ** c2.index) * 1e4,
    "our ZC fwd (%)": c2["fwd_zc_pct"],
})
parity_mkt["ICAP-implied ZC fwd (%)"] = ((1.02 ** parity_mkt.index + parity_mkt.iloc[:, 0] / 1e4)
                                         ** (1 / parity_mkt.index) - 1) * 100
parity_mkt["ZC fwd gap (bp)"] = (parity_mkt["ICAP-implied ZC fwd (%)"] - parity_mkt["our ZC fwd (%)"]) * 100
for model in ZC_MODELS:
    parity_mkt[f"{model} vol cap/floor 2%"] = (c2[f"vol_{model}"].round(2).astype(str) + " / "
                                               + f2[f"vol_{model}"].round(2).astype(str))
print("Put-call parity at 2%")
display(parity_mkt.round(3))

# Smiles
fig = make_subplots(rows=1, cols=2, subplot_titles=["Bachelier: normal vol of ZC rate (bp)",
                                                    "Black: lognormal vol of index ratio (%)"])
colors = px.colors.qualitative.Plotly
for j, T in enumerate(SURFACES["bachelier"].index):
    for col, model in enumerate(ZC_MODELS, start=1):
        s = SURFACES[model]
        fig.add_scatter(x=s.columns, y=s.loc[T], mode="lines+markers", name=f"{T}Y", legendgroup=f"{T}Y",
                        showlegend=(col == 1), line=dict(color=colors[j % len(colors)]), row=1, col=col)
fig.update_xaxes(title_text="strike (%)")
fig.update_layout(title="ICAP-implied ZC vol smiles", height=480, hovermode="x unified")
fig.show()

### 7.3 Repricing caps/floors with the implied vols

`zc_option(...)` prices any maturity/strike with either model, reading the vol from that model's calibrated surface
(linear in maturity then strike, flat extrapolation). Checks:
1. **Market quotes repriced**: difference vs ICAP ≈ 0 for both models (except at 2%, where the surface holds the
   cap/floor average).
2. **Off-grid strikes/maturities**: both models agree on the grid but not necessarily between quotes. The gap is
   interpolation / model risk.
3. **Put-call parity** with the swap, **strike strip** vs market points, **greeks**.

In [ ]:
def surface_vol(surface: pd.DataFrame, T, strike_pct) -> float:
    """Linear in maturity, then linear in strike, flat extrapolation (surface in display units)."""
    ts = surface.index.to_numpy(float)
    by_strike = [np.interp(T, ts, surface[k].to_numpy(float)) for k in surface.columns]
    return float(np.interp(strike_pct, surface.columns.to_numpy(float), by_strike))


def zc_vol(T, strike_pct, model="bachelier") -> float:
    """Model vol in decimal from the calibrated surface."""
    return surface_vol(SURFACES[model], T, strike_pct) / VOL_UNITS[model][1]


def zc_option(notional, T, strike_pct, kind="cap", model="bachelier", vol=None, infl=None, disc=None) -> dict:
    """ZC cap/floor: vol (decimal, model units) from the calibrated surface unless given."""
    infl = inf_curve if infl is None else infl
    disc = ois_curve if disc is None else disc
    is_cap = kind == "cap"
    F_ratio = zc_index_ratio(T, infl)
    K_ratio = (1 + strike_pct / 100) ** T
    vol = zc_vol(T, strike_pct, model) if vol is None else vol
    df = disc.df(T)
    fwd = zc_fwd_premium(F_ratio, K_ratio, T, vol, model, is_cap)
    F_r = F_ratio ** (1 / T) - 1
    fwd_up = zc_fwd_premium((1 + F_r + 1e-4) ** T, K_ratio, T, vol, model, is_cap)
    dvol = 1e-4 if model == "bachelier" else 1e-2          # 1bp normal / 1% lognormal
    fwd_vup = zc_fwd_premium(F_ratio, K_ratio, T, vol + dvol, model, is_cap)
    unit, scale = VOL_UNITS[model]
    return {"model": model, "kind": kind, "T": T, "strike_pct": strike_pct, "fwd_zc_pct": F_r * 100,
            f"vol": vol * scale, "vol_unit": unit, "fwd_premium_bp": fwd * 1e4, "spot_premium_bp": fwd * df * 1e4,
            "pv": notional * df * fwd, "delta_per_bp": notional * df * (fwd_up - fwd),
            "vega_per_vol_pt": notional * df * (fwd_vup - fwd)}


# 1) Reprice every ICAP quote with both models (in the calibration units: fwd or spot)
col = "fwd_premium_bp" if ZC_OPT_PREMIUM_TYPE == "FWD" else "spot_premium_bp"
for model in ZC_MODELS:
    opt[f"reprice_{model}"] = [zc_option(1, r.tenor_y, r.strike_pct, r.kind, model)[col] for r in opt.itertuples()]
    opt[f"diff_{model}"] = opt[f"reprice_{model}"] - opt["PREMIUM_BP"]
summary = pd.DataFrame({m: {"max |diff| all (bp)": opt[f"diff_{m}"].abs().max(),
                            "max |diff| excl. 2% (bp)": opt.loc[opt["strike_pct"] != 2.0, f"diff_{m}"].abs().max()}
                        for m in ZC_MODELS})
print(f"Repricing ICAP {ZC_OPT_PREMIUM_TYPE} premiums with the implied surfaces")
display(summary.round(4))
opt.pivot_table(index="tenor_y", columns=["kind", "strike_pct"], values="diff_bachelier").round(3)

In [ ]:
# 2) Off-grid prices: both models
offgrid = [(4, 1.25, "floor"), (8, 0.25, "floor"), (8, 2.25, "cap"), (12, 1.75, "floor"), (12, 3.25, "cap"),
           (18, 2.75, "cap"), (10, round(zc_option(1, 10, 2.0)["fwd_zc_pct"], 3), "cap")]
rows = []
for T, k, kind in offgrid:
    b = zc_option(10e6, T, k, kind, "bachelier")
    l = zc_option(10e6, T, k, kind, "black")
    rows.append({"T": T, "strike_pct": k, "kind": kind, "vol bachelier (bp)": b["vol"], "vol black (%)": l["vol"],
                 "spot bp bachelier": b["spot_premium_bp"], "spot bp black": l["spot_premium_bp"],
                 "model gap (bp)": l["spot_premium_bp"] - b["spot_premium_bp"],
                 "PV bachelier (10m)": b["pv"], "PV black (10m)": l["pv"]})
print("Off-grid options, EUR 10m (last row = 10Y ATM)")
pd.DataFrame(rows).round(2)

In [ ]:
# 3) Put-call parity with the ZC swap, both models
parity = []
for T, K in [(5, 1.0), (10, 2.0), (10, 3.0), (20, 2.5), (30, 2.0)]:
    swp = price_zc_swap(10e6, T, K, receive_inflation=True)["npv"]
    for model in ZC_MODELS:
        cap = zc_option(10e6, T, K, "cap", model)["pv"]
        flo = zc_option(10e6, T, K, "floor", model)["pv"]
        parity.append({"T": T, "K": K, "model": model, "cap": cap, "floor": flo, "cap - floor": cap - flo,
                       "ZC swap NPV": swp, "diff": cap - flo - swp})
pd.DataFrame(parity).round(2)

In [ ]:
# Strike strip vs market (forward premiums), chosen maturity
T_strip = 10
ks = np.round(np.arange(-1.0, 5.01, 0.1), 2)
fig = go.Figure()
for model, dash in [("bachelier", None), ("black", "dash")]:
    for kind in ["floor", "cap"]:
        y = [zc_option(1, T_strip, k, kind, model)["fwd_premium_bp"] for k in ks]
        fig.add_scatter(x=ks, y=y, mode="lines", name=f"{kind} - {model}", line=dict(dash=dash))
mk = opt[opt["tenor_y"] == T_strip]
for kind, sym in [("floor", "circle"), ("cap", "diamond")]:
    m = mk[mk["kind"] == kind]
    fig.add_scatter(x=m["strike_pct"], y=m["MKT_FWD_BP"], mode="markers", name=f"ICAP {kind}",
                    marker=dict(size=10, symbol=sym, color="black"))
fig.update_layout(title=f"{T_strip}Y ZC caps/floors repriced with implied vols vs ICAP (forward premium)",
                  xaxis_title="strike (%)", yaxis_title="forward premium (bp)", hovermode="x unified", height=480)
fig.show()

# Greeks, EUR 10m
greeks = pd.DataFrame([zc_option(10e6, T, k, kind, model)
                       for T in [5, 10] for k in [1.0, 2.0, 3.0] for kind in ["cap", "floor"] for model in ZC_MODELS])
greeks[["model", "kind", "T", "strike_pct", "fwd_zc_pct", "vol", "vol_unit", "spot_premium_bp", "pv",
        "delta_per_bp", "vega_per_vol_pt"]].round(2)

### 7.4 Payoff vs option price

**What is "spot" for a ZC inflation option?** The underlying is the realised cumulative inflation
$X = I(B+12T)/I(B)$; before expiry the tradable proxy is its forward, i.e. the **ZC breakeven to $T$**,
$F_r = F^{1/T} - 1$ (the ZC swap rate of that maturity). All profiles below use $F_r$ on the x-axis.

**Time convention**: the option keeps its maturity $T$ (payoff on the full period), only the *remaining* uncertainty
$\tau$ shrinks: Bachelier $s = \sigma_N\,T(1+F_r)^{T-1}\sqrt{\tau}$, Black $s = \sigma\sqrt{\tau}$. At $\tau = 0$ the value
is the payoff. Values are **forward** (paid at $T$, undiscounted, € on the notional); PV today = value × $P(0,T)$.

**Compounding convexity**: even at expiry the payoff is *not* linear in the rate:
$\text{cap} = N\max\left((1+r)^T - (1+K)^T, 0\right)$ has slope $N\,T(1+r)^{T-1}$, which increases with $r$.
In the money, a 10Y cap on 10m moves ~€10-12k per bp, like a ZC swap (its IE01).

**Vol in scenarios**: *sticky strike*, the vol is read on the calibrated surface at the option's own strike and
kept constant when the forward moves.

`POSITIONS` is your book: add/remove lines (long `side=+1`, short `side=-1`).

In [ ]:
# ---- Book of ZC options (edit) ------------------------------------------------
GREEKS_MODEL = "bachelier"     # "bachelier" or "black"
POSITIONS = [
    dict(kind="cap",   strike_pct=2.0, T=10, notional=10e6, side=+1),
    # dict(kind="floor", strike_pct=2.0, T=10, notional=10e6, side=+1),   # + the cap above = straddle
    # dict(kind="cap",   strike_pct=3.0, T=10, notional=10e6, side=-1),   # cap spread 2% / 3%
    # dict(kind="floor", strike_pct=0.0, T=10, notional=10e6, side=-1),   # short deflation floor
]
# -------------------------------------------------------------------------------


def option_fwd_value(F_r, pos, tau, model=GREEKS_MODEL, vol=None, vol_shift=0.0):
    """Forward value (EUR, paid at T) of one position, for a ZC forward rate F_r (decimal, scalar or array)
    and remaining time tau (years). Sticky-strike vol from the calibrated surface unless given."""
    F_r = np.asarray(F_r, float)
    T, k = pos["T"], pos["strike_pct"]
    is_cap = pos["kind"] == "cap"
    K_ratio = (1 + k / 100) ** T
    F_ratio = (1 + F_r) ** T
    vol = (zc_vol(T, k, model) if vol is None else vol) + vol_shift
    if tau <= 0:
        px = np.maximum(F_ratio - K_ratio, 0) if is_cap else np.maximum(K_ratio - F_ratio, 0)
    elif model == "bachelier":
        sd = vol * T * (1 + F_r) ** (T - 1) * np.sqrt(tau)
        d = (F_ratio - K_ratio) / sd
        px = ((F_ratio - K_ratio) * norm.cdf(d) + sd * norm.pdf(d) if is_cap
              else (K_ratio - F_ratio) * norm.cdf(-d) + sd * norm.pdf(d))
    else:
        sd = vol * np.sqrt(tau)
        d1 = (np.log(F_ratio / K_ratio) + 0.5 * sd ** 2) / sd
        d2 = d1 - sd
        px = (F_ratio * norm.cdf(d1) - K_ratio * norm.cdf(d2) if is_cap
              else K_ratio * norm.cdf(-d2) - F_ratio * norm.cdf(-d1))
    return pos["side"] * pos["notional"] * px


def book_value(shift_bp, tau_frac, positions=POSITIONS, model=GREEKS_MODEL, vol_shift=0.0):
    """Book forward value for a parallel shift of ZC breakevens (bp) at a fraction of each option's life left."""
    shift = np.asarray(shift_bp, float) / 1e4
    total = 0.0
    for pos in positions:
        F_r0 = zc_index_ratio(pos["T"]) ** (1 / pos["T"]) - 1
        total = total + option_fwd_value(F_r0 + shift, pos, tau_frac * pos["T"], model, vol_shift=vol_shift)
    return total


def describe(pos):
    return f"{'long' if pos['side'] > 0 else 'short'} {pos['T']}Y {pos['kind']} {pos['strike_pct']:.2f}% x {pos['notional']/1e6:.0f}m"


print("Book:", " | ".join(describe(p_) for p_ in POSITIONS), f"| model {GREEKS_MODEL}")

In [ ]:
# Figure 1 - single 10Y cap and floor at 2%: payoff at expiry vs value with tau years left
T_demo, K_demo = 10, 2.0
F_r0 = zc_index_ratio(T_demo) ** (1 / T_demo) - 1
rates = np.linspace(-0.01, 0.05, 301)
taus = [T_demo, 5, 2, 0.5, 0.0]

fig = make_subplots(rows=1, cols=2, subplot_titles=[f"{T_demo}Y ZC cap {K_demo}%", f"{T_demo}Y ZC floor {K_demo}%"])
palette = px.colors.sequential.Viridis
for col, kind in enumerate(["cap", "floor"], start=1):
    pos = dict(kind=kind, strike_pct=K_demo, T=T_demo, notional=10e6, side=+1)
    for j, tau in enumerate(taus):
        y = option_fwd_value(rates, pos, tau)
        name = "payoff at expiry" if tau == 0 else f"value, {tau:g}y left"
        fig.add_scatter(x=rates * 100, y=y, mode="lines", name=name, legendgroup=name, showlegend=(col == 1),
                        line=dict(color="black" if tau == 0 else palette[int(j * (len(palette) - 1) / len(taus))],
                                  width=3 if tau in (0, T_demo) else 1.5,
                                  dash="dash" if tau == 0 else None), row=1, col=col)
    fig.add_vline(x=K_demo, line_dash="dot", line_color="red", row=1, col=col)
    fig.add_vline(x=F_r0 * 100, line_dash="dot", line_color="green", row=1, col=col)
fig.update_xaxes(title_text="ZC breakeven to maturity (%)  [red: strike, green: today's forward]")
fig.update_yaxes(title_text="forward value (EUR on 10m)", col=1)
fig.update_layout(title=f"Payoff vs option value ({GREEKS_MODEL}, sticky-strike vol)", height=500, hovermode="x unified")
fig.show()

# Time value = value - intrinsic (on today's forward)
for kind in ["cap", "floor"]:
    pos = dict(kind=kind, strike_pct=K_demo, T=T_demo, notional=10e6, side=+1)
    v, intr = option_fwd_value(F_r0, pos, T_demo), option_fwd_value(F_r0, pos, 0)
    print(f"{T_demo}Y {kind} {K_demo}% @ fwd {F_r0*100:.3f}%: value {v:,.0f}  intrinsic {intr:,.0f}  "
          f"time value {v - intr:,.0f}  (PV today = x {ois_curve.df(T_demo):.4f})")

In [ ]:
# Figure 2 - your book (POSITIONS): payoff vs value, x = parallel shift of ZC breakevens
shifts = np.linspace(-300, 300, 301)
same_T = len({p_["T"] for p_ in POSITIONS}) == 1
x0 = (zc_index_ratio(POSITIONS[0]["T"]) ** (1 / POSITIONS[0]["T"]) - 1) * 100 if same_T else 0.0
x = x0 + shifts / 100 if same_T else shifts
fig = go.Figure()
for frac, name, style in [(1.0, "value today", dict(width=3)), (0.5, "value, half life left", dict(width=1.5)),
                          (0.1, "value, 10% life left", dict(width=1.5)), (0.0, "payoff at expiry", dict(width=3, dash="dash", color="black"))]:
    fig.add_scatter(x=x, y=book_value(shifts, frac), mode="lines", name=name, line=style)
for p_ in POSITIONS:
    fig.add_vline(x=p_["strike_pct"] if same_T else (p_["strike_pct"] - x0) * 100, line_dash="dot", line_color="red")
fig.add_vline(x=x0 if same_T else 0, line_dash="dot", line_color="green")
fig.add_hline(y=0, line_color="gray")
fig.update_layout(title="Book: " + " | ".join(describe(p_) for p_ in POSITIONS),
                  xaxis_title="ZC breakeven (%)" if same_T else "parallel shift of ZC breakevens (bp)",
                  yaxis_title="forward value (EUR)", hovermode="x unified", height=480)
fig.show()

### 7.5 Greeks: delta, gamma, vega

Definitions used (finite differences on the full pricer, forward €, per position notional):
- **Delta** = $\partial V / \partial F_r$ per **1 bp** of ZC breakeven (€/bp). Deep ITM cap → $N\,T(1+F_r)^{T-1}\times 10^{-4}$, the ZC swap IE01.
- **Gamma** = change in delta for a 1 bp move (€/bp²). Peaks ATM, grows as $\tau \to 0$ (pin risk). Note it is not zero deep ITM: the compounding convexity $N\,T(T-1)(1+F_r)^{T-2}$ gives a small residual gamma, also present in the ZC swap.
- **Vega** = $\partial V / \partial \sigma$ per **1 bp** of normal vol (Bachelier) or **1%** of lognormal vol (Black). Peaks ATM, scales with $\sqrt{\tau}$.

**Bachelier analytics** (on the ratio $X$, $s = \sigma_N\,T(1+F_r)^{T-1}\sqrt\tau$, $d = (F-\bar K)/s$):
$\frac{\partial V}{\partial F} = \Phi(d)$ (cap), $\frac{\partial^2 V}{\partial F^2} = \frac{\varphi(d)}{s}$,
$\frac{\partial V}{\partial s} = \varphi(d)$. In rate terms, since both $F = (1+F_r)^T$ and $s$ depend on $F_r$:
$$\Delta_r = \Phi(d)\,T(1+F_r)^{T-1} + \varphi(d)\,\frac{\partial s}{\partial F_r}, \qquad
\text{Vega}_{\sigma_N} = \varphi(d)\,T(1+F_r)^{T-1}\sqrt\tau$$
The second delta term is a *sticky-normal-rate-vol* effect: a higher breakeven increases the vol of the index ratio.
Checked against the finite differences in the table below.

In [ ]:
def position_greeks(F_r, pos, tau, model=GREEKS_MODEL, h_bp=1.0):
    """Delta (EUR/bp), gamma (EUR/bp^2), vega (EUR per 1bp normal or 1% lognormal) by finite differences."""
    h = h_bp / 1e4
    v0 = option_fwd_value(F_r, pos, tau, model)
    vu = option_fwd_value(F_r + h, pos, tau, model)
    vd = option_fwd_value(F_r - h, pos, tau, model)
    dv = 1e-4 if model == "bachelier" else 1e-2
    vvu = option_fwd_value(F_r, pos, tau, model, vol_shift=dv)
    vvd = option_fwd_value(F_r, pos, tau, model, vol_shift=-dv)
    return {"value": v0, "delta": (vu - vd) / (2 * h_bp), "gamma": (vu - 2 * v0 + vd) / h_bp ** 2, "vega": (vvu - vvd) / 2}


def book_greeks(shift_bp, tau_frac, positions=POSITIONS, model=GREEKS_MODEL):
    out = {"value": 0.0, "delta": 0.0, "gamma": 0.0, "vega": 0.0}
    for pos in positions:
        F_r0 = zc_index_ratio(pos["T"]) ** (1 / pos["T"]) - 1
        g = position_greeks(F_r0 + np.asarray(shift_bp, float) / 1e4, pos, tau_frac * pos["T"], model)
        out = {k_: out[k_] + g[k_] for k_ in out}
    return out


# Analytic vs finite-difference check (Bachelier, single 10Y cap 2%, today)
pos = dict(kind="cap", strike_pct=2.0, T=10, notional=10e6, side=+1)
T_, tau_ = pos["T"], pos["T"]
F_r = zc_index_ratio(T_) ** (1 / T_) - 1
sig = zc_vol(T_, pos["strike_pct"], "bachelier")
F, Kb = (1 + F_r) ** T_, (1 + pos["strike_pct"] / 100) ** T_
dX = T_ * (1 + F_r) ** (T_ - 1)
s = sig * dX * np.sqrt(tau_)
ds = sig * T_ * (T_ - 1) * (1 + F_r) ** (T_ - 2) * np.sqrt(tau_)
d = (F - Kb) / s
fd = position_greeks(F_r, pos, tau_, "bachelier", h_bp=0.01)
pd.DataFrame({
    "analytic": {"delta (EUR/bp)": pos["notional"] * (norm.cdf(d) * dX + norm.pdf(d) * ds) * 1e-4,
                 "vega (EUR/bp vol)": pos["notional"] * norm.pdf(d) * dX * np.sqrt(tau_) * 1e-4},
    "finite diff": {"delta (EUR/bp)": fd["delta"], "vega (EUR/bp vol)": fd["vega"]},
}).round(2)

In [ ]:
# Figure 3 - book greeks vs ZC breakeven, several remaining lives (+ Black today, dashed, for delta & gamma)
shifts = np.linspace(-300, 300, 601)
x = x0 + shifts / 100 if same_T else shifts
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=["Delta (EUR per bp)", "Gamma (EUR per bp²)",
                                    f"Vega (EUR per {'1bp normal' if GREEKS_MODEL == 'bachelier' else '1% lognormal'} vol)"])
for j, (frac, name) in enumerate([(1.0, "today"), (0.5, "half life left"), (0.1, "10% life left"), (0.02, "2% life left")]):
    g = book_greeks(shifts, frac)
    color = palette[int(j * (len(palette) - 1) / 4)]
    for row, key in enumerate(["delta", "gamma", "vega"], start=1):
        fig.add_scatter(x=x, y=g[key], mode="lines", name=name, legendgroup=name, showlegend=(row == 1),
                        line=dict(color=color, width=3 if frac == 1.0 else 1.5), row=row, col=1)
other = "black" if GREEKS_MODEL == "bachelier" else "bachelier"
g_other = book_greeks(shifts, 1.0, model=other)
for row, key in enumerate(["delta", "gamma"], start=1):
    fig.add_scatter(x=x, y=g_other[key], mode="lines", name=f"today ({other})", legendgroup="other",
                    showlegend=(row == 1), line=dict(color="red", dash="dash"), row=row, col=1)
for p_ in POSITIONS:
    for row in (1, 2, 3):
        fig.add_vline(x=p_["strike_pct"] if same_T else (p_["strike_pct"] - x0) * 100, line_dash="dot",
                      line_color="gray", row=row, col=1)
fig.update_xaxes(title_text="ZC breakeven (%)" if same_T else "parallel shift (bp)", row=3, col=1)
fig.update_layout(title=f"Book greeks vs breakeven ({GREEKS_MODEL}, sticky strike)", height=850, hovermode="x unified")
fig.show()

In [ ]:
# Figure 4 - greeks vs remaining life for ITM / ATM / OTM 10Y caps (Bachelier, forward fixed)
T_c = 10
F_r0 = zc_index_ratio(T_c) ** (1 / T_c) - 1
cases = {"ITM (K = fwd - 50bp)": F_r0 * 100 - 0.5, "ATM (K = fwd)": F_r0 * 100, "OTM (K = fwd + 50bp)": F_r0 * 100 + 0.5}
tau_grid = np.linspace(T_c, 0.01, 200)
fig = make_subplots(rows=1, cols=3, subplot_titles=["Delta (EUR/bp)", "Gamma (EUR/bp²)", "Vega (EUR/bp vol)"])
for j, (name, k) in enumerate(cases.items()):
    pos = dict(kind="cap", strike_pct=round(k, 4), T=T_c, notional=10e6, side=+1)
    gs = [position_greeks(F_r0, pos, t_, "bachelier") for t_ in tau_grid]
    for col, key in enumerate(["delta", "gamma", "vega"], start=1):
        fig.add_scatter(x=tau_grid, y=[g_[key] for g_ in gs], mode="lines", name=name, legendgroup=name,
                        showlegend=(col == 1), line=dict(color=px.colors.qualitative.Plotly[j]), row=1, col=col)
fig.update_xaxes(title_text="remaining life (years)", autorange="reversed")
fig.update_layout(title=f"{T_c}Y ZC cap on 10m: greeks as time passes (forward unchanged)", height=430, hovermode="x unified")
fig.show()

### 7.6 The linker's embedded deflation floor

OAT€i redemption = $100\max(IR(T), 1)$ = $100\,IR(T)$ + $100\max(1 - IR(T), 0)$. With
$IR(T) = IR(s)\,X$, $X = \frac{\text{Ref}(T)}{\text{Ref}(s)}$, the floor is $100\,IR(s)\max\left(\frac{1}{IR(s)} - X, 0\right)$:
a ZC floor with strike ratio $1/IR(s)$, priced with the calibrated surfaces (vol read at the equivalent annual
strike; flat extrapolation below −1%). For the 2036 line ($IR \approx 1.31$) the strike is deep out of the money;
for a **new issue** ($IR = 1$) it's a 0% ZC floor.

In [ ]:
def linker_floor_value(ir_settle, T, F_ratio, df, model="bachelier") -> dict:
    K_ratio = 1 / ir_settle
    k_pct = (K_ratio ** (1 / T) - 1) * 100
    vol = zc_vol(T, k_pct, model)
    fwd = zc_fwd_premium(F_ratio, K_ratio, T, vol, model, is_cap=False)
    return {"model": model, "IR_settle": ir_settle, "strike_pct_pa": k_pct, "fwd_pct_pa": (F_ratio ** (1 / T) - 1) * 100,
            "vol": vol * VOL_UNITS[model][1], "floor_value_per_100": 100 * ir_settle * df * fwd}


T_lk = (lk["maturity"] - lk["settle"]).days / 365.0
F_lk = ref_cpi(lk["maturity"]) / ref_cpi(lk["settle"])
df_lk = ois_curve.df(yf(lk["maturity"])) / ois_curve.df(yf(lk["settle"]))
pd.DataFrame({f"{name} - {m}": linker_floor_value(ir, T_lk, F_lk, df_lk, m)
              for name, ir in [("OAT€i 2036", lk["ir_settle"]), ("new 10Y issue", 1.0)] for m in ZC_MODELS}).round(6)

### 7.7 YoY caps/floors

Caplet $i$ pays at $T_i$: $N\max(Y_i - K, 0)$ with $Y_i = \frac{I(T_i)}{I(T_{i-1})} - 1$. Bachelier on the
**convexity-adjusted** forward YoY (4.4), expiry $T_i$. No YoY broker quotes yet, so the YoY normal vols are
**example inputs**. Parity: YoY cap − YoY floor = YoY swap receiving inflation at $K$.

In [ ]:
# ---- YoY vols (EXAMPLE values, normal vols in bp) ---------------------------
YOY_NORMAL_VOLS_BP = {1: 110, 2: 100, 3: 95, 5: 90, 7: 88, 10: 85, 15: 82, 20: 80, 30: 78}
# -----------------------------------------------------------------------------


def interp_vol(table, T) -> float:
    k = np.array(sorted(table), float)
    return float(np.interp(T, k, [table[int(x)] for x in k])) / 1e4


def yoy_cap_floor(notional, n_years, strike_pct, kind="cap", params=CONVEXITY_PARAMS, adjusted=True):
    f = yoy_forwards(n_years, params)
    K = strike_pct / 100
    rows = []
    for i, r in f.iterrows():
        F = (r["yoy_adjusted_pct"] if adjusted else r["yoy_linear_pct"]) / 100
        sigma = interp_vol(YOY_NORMAL_VOLS_BP, i)
        pv = notional * r["df"] * bachelier(F, K, sigma * np.sqrt(i), kind == "cap")
        rows.append({"period": i, "fwd_yoy_pct": F * 100, "vol_bp": sigma * 1e4, "df": r["df"], "pv": pv})
    caplets = pd.DataFrame(rows).set_index("period")
    return caplets, caplets["pv"].sum()


caplets, cap10 = yoy_cap_floor(10e6, 10, 2.0, "cap")
floorlets, floor10 = yoy_cap_floor(10e6, 10, 2.0, "floor")
f10 = yoy_forwards(10)
yoy_swap_pv = (10e6 * f10["df"] * (f10["yoy_adjusted_pct"] / 100 - 0.02)).sum()
print(f"10Y YoY cap 2%  : {cap10:,.0f}")
print(f"10Y YoY floor 2%: {floor10:,.0f}")
print(f"cap - floor     : {cap10 - floor10:,.0f}   vs YoY swap (rec. infl @2%): {yoy_swap_pv:,.0f}")
_, cap10_lin = yoy_cap_floor(10e6, 10, 2.0, "cap", adjusted=False)
print(f"cap on linear (unadjusted) forwards: {cap10_lin:,.0f}  -> convexity worth {cap10 - cap10_lin:,.0f}")
caplets.join(floorlets["pv"].rename("floorlet_pv")).rename(columns={"pv": "caplet_pv"}).round(4)